# Forecast Validation Against Official Statistics Botswana Data
**Forecasting Botswana's Export Diversification Opportunities**
*Boemo Seboni | DSS 680 – Summer 2026*

The models in this project were trained on UN Comtrade data through 2023. After modeling was complete, Statistics Botswana published official trade figures for **2024 and 2025**. Because these figures were never used in training, validation, or model selection, they provide an independent test of how the forecasts held up against what actually happened.

Statistics Botswana reports trade in less detail than UN Comtrade, so forecasts are compared at three levels: **total exports**, **diamond exports**, and **non-diamond exports**.

**Workflow**
1. Download the official Statistics Botswana *International Merchandise Trade Statistics, December 2025* digest (Steps 1–2)
2. Extract and clean the export tables from the PDF, reconciling them against published totals (Steps 3–7)
3. Convert values from Botswana pula to US dollars using official Bank of Botswana exchange rates (Step 8)
4. Save the cleaned actual figures (Step 9)
5. Compare the SAS forecasts with the actual figures (Steps 10–12)

**Inputs:** `botswana_complete_master_2000_2023.csv` (from the pipeline notebook), plus `table_LASSOSCOREDATA.csv` and `table_FINALPROJECTSCOREDATA.csv` (LASSO and SVM scores exported from SAS Viya)

*Code developed with AI assistance.*

## Step 1 — Set up and download the source publication
Creates the project folders, downloads the December 2025 digest from Statistics Botswana (if it hasn't already been downloaded), and confirms the PDF is readable.

In [1]:
# ============================================================
# STEP 1: SET UP THE STATISTICS BOTSWANA VALIDATION NOTEBOOK
# ============================================================

from pathlib import Path
import requests
import pandas as pd
import numpy as np
import pdfplumber

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ------------------------------------------------------------
# Project folders
# ------------------------------------------------------------

PROJECT_FOLDER = Path("statistics_botswana_validation")
RAW_FOLDER = PROJECT_FOLDER / "raw"
OUTPUT_FOLDER = PROJECT_FOLDER / "output"

RAW_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Official Statistics Botswana source
# ------------------------------------------------------------

STATSBOTS_DEC_2025_URL = (
    "https://www.statsbots.org.bw/sites/default/files/publications/"
    "International%20Merchandise%20Trade%20Statistics%20"
    "Monthly%20Digest-December%202025.pdf"
)

STATSBOTS_DEC_2025_FILE = (
    RAW_FOLDER / "statistics_botswana_imts_december_2025.pdf"
)

# ------------------------------------------------------------
# Download the consolidated publication
# ------------------------------------------------------------

if not STATSBOTS_DEC_2025_FILE.exists():
    response = requests.get(
        STATSBOTS_DEC_2025_URL,
        timeout=120,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()
    STATSBOTS_DEC_2025_FILE.write_bytes(response.content)

# ------------------------------------------------------------
# Verify that Python received a readable PDF
# ------------------------------------------------------------

with pdfplumber.open(STATSBOTS_DEC_2025_FILE) as pdf:
    page_count = len(pdf.pages)
    first_page_text = pdf.pages[0].extract_text() or ""

file_size_mb = STATSBOTS_DEC_2025_FILE.stat().st_size / (1024 ** 2)

print("Statistics Botswana source downloaded successfully")
print(f"File: {STATSBOTS_DEC_2025_FILE}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Number of pages: {page_count}")
print(
    "Expected publication found:",
    "December 2025" in first_page_text
    or "DECEMBER 2025" in first_page_text.upper()
)

Statistics Botswana source downloaded successfully
File: statistics_botswana_validation\raw\statistics_botswana_imts_december_2025.pdf
File size: 2.40 MB
Number of pages: 34
Expected publication found: True


## Step 2 — Inventory every table in the PDF
Extracts every table pdfplumber can detect and stores it by page and table number, so later steps can find specific tables.

In [2]:
# ============================================================
# STEP 2: INVENTORY EVERY DETECTABLE TABLE IN THE PDF
# ============================================================

import pdfplumber
import pandas as pd

pdf_file = STATSBOTS_DEC_2025_FILE

table_settings = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "snap_tolerance": 4,
    "join_tolerance": 4,
    "intersection_tolerance": 5
}

# Store every raw table so nothing is discarded
all_pdf_tables = {}

inventory_records = []

with pdfplumber.open(pdf_file) as pdf:

    for page_number, page in enumerate(pdf.pages, start=1):

        page_text = page.extract_text() or ""
        page_lines = [
            line.strip()
            for line in page_text.splitlines()
            if line.strip()
        ]

        # Capture titles or labels appearing on the page
        table_labels = [
            line for line in page_lines
            if (
                line.upper().startswith("TABLE")
                or "TABLE " in line.upper()
            )
        ]

        detected_tables = page.extract_tables(table_settings)

        for table_number, table in enumerate(detected_tables, start=1):

            table_key = f"page_{page_number}_table_{table_number}"
            all_pdf_tables[table_key] = table

            row_count = len(table)

            column_count = max(
                (len(row) for row in table if row is not None),
                default=0
            )

            first_nonempty_rows = []

            for row in table:
                if row and any(
                    value not in {None, ""}
                    for value in row
                ):
                    first_nonempty_rows.append(row)

                if len(first_nonempty_rows) == 3:
                    break

            inventory_records.append({
                "table_key": table_key,
                "pdf_page": page_number,
                "table_number_on_page": table_number,
                "number_of_rows": row_count,
                "number_of_columns": column_count,
                "page_table_labels": " | ".join(table_labels),
                "first_row_preview": str(
                    first_nonempty_rows[0]
                    if len(first_nonempty_rows) > 0
                    else ""
                )[:250],
                "second_row_preview": str(
                    first_nonempty_rows[1]
                    if len(first_nonempty_rows) > 1
                    else ""
                )[:250]
            })

table_inventory = pd.DataFrame(inventory_records)

print("PDF pages inspected:", page_number)
print("Tables detected:", len(table_inventory))
print("Raw tables preserved:", len(all_pdf_tables))

display(
    table_inventory[
        [
            "table_key",
            "pdf_page",
            "number_of_rows",
            "number_of_columns",
            "page_table_labels",
            "first_row_preview"
        ]
    ]
)

PDF pages inspected: 34
Tables detected: 22
Raw tables preserved: 22


,table_key,pdf_page,number_of_rows,number_of_columns,page_table_labels,first_row_preview
0,page_9_table_1,9,5,13,respectively. The increase in total exports is...,['TABLE A: Data Revision: November 2025 (Milli...
1,page_10_table_1,10,1,2,"November 2025 digest to P7, 135.4 million in t...","['J F M A M J A S O N D\n2024', 'J F M A M J A..."
2,page_11_table_1,11,6,14,Note: ‘Other Goods’ here includes any other go...,
3,page_11_table_2,11,10,7,Note: ‘Other Goods’ here includes any other go...,['70.0\n60.0\n50.0\n40.0 )%(\nNOITUBIRTNOC\n30...
4,page_13_table_1,13,7,4,‘Rest of the World’ includes all other Countri...,"['', '', '', '8.2']"
5,page_14_table_1,14,10,12,Rest of the World’ includes all other Countrie...,
6,page_15_table_1,15,7,5,in Botswana for movement of international merc...,['TABLE B: Trade by Mode of Transport – Decemb...
7,page_16_table_1,16,11,3,"583.0 million) respectively, as displayed in T...",['TABLE C1: Transit Trade by Border Post-Decem...
8,page_18_table_1,18,17,5,TABLE C2: Transit Trade by Chapter- December 2...,['TABLE C2: Transit Trade by Chapter- December...
9,page_18_table_2,18,9,8,TABLE C2: Transit Trade by Chapter- December 2...,"['', '', '', '', '6.1', None, None, '']"


A quick preview of the first rows of Table 2.2 (Principal Exports Commodity Groups) as extracted from the PDF:

In [3]:
all_pdf_tables["page_21_table_1"][:5]

[['TABLE 2.2: Principal Exports Commodity Groups – January 2024 to December 2025 (Million Pula)',
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None],
 ['SH\n\\\ndoireP',
  'laoC',
  'reppoC',
  'sdnomaiD',
  'dloG',
  'leetS\nstcudorP\n&\nnorI',
  'elttaC\neviL',
  '&\nyrenihcaM tnempiuqE\nlacirtcelE',
  'taeM\nstcudorP\n&\ntaeM',
  'citsalP\n& stcudorP\ncitsalP',
  'hsA\nadoS\n&\ntlaS',
  'selitxeT',
  'tnempiuqE\n& tropsnarT\nselciheV',
  'sdooG\nrehtO',
  'sdooG\nlatoT'],
 ['Jan_2024',
  '25.8',
  '885.7',
  '4,733.1',
  '20.5',
  '14.7',
  '32.2',
  '214.2',
  '21.4',
  '26.0',
  '141.8',
  '27.2',
  '71.2',
  '85.9',
  '6,299.8'],
 ['Feb',
  '23.1',
  '674.4',
  '3,184.0',
  '6.1',
  '23.6',
  '42.5',
  '309.6',
  '16.5',
  '33.5',
  '87.9',
  '28.3',
  '28.8',
  '99.1',
  '4,557.2'],
 ['Mar',
  '26.1',
  '775.0',
  '5,335.4',
  '3.4',
  '33.3',
  '50.0',
  '241.3',
  '38.4',
  '39.7',
  '131.6',
  '28.3',
  '24.7',
  '1

## Step 3 — Clean Table 2.2: Principal export commodity groups
The PDF layout merges some rows together during extraction. This step separates them and converts the values to numbers.

In [5]:
# ============================================================
# STEP 3: CLEAN TABLE 2.2 AND SEPARATE MERGED PDF ROWS
# Principal Exports Commodity Groups, 2024–2025
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Retrieve the raw table preserved during Step 2
# ------------------------------------------------------------

table_22_key = "page_21_table_1"
raw_table_22 = all_pdf_tables[table_22_key]

table_title = str(raw_table_22[0][0])

assert "TABLE 2.2" in table_title.upper(), (
    f"Unexpected table selected: {table_title}"
)

print("Selected table:")
print(table_title)

# ------------------------------------------------------------
# Assign readable column names
#
# The PDF's vertically rotated headings are extracted backwards.
# These names identify the columns; all numerical observations
# continue to come directly from the PDF.
# ------------------------------------------------------------

column_names = [
    "period",
    "coal_bwp_million",
    "copper_bwp_million",
    "diamonds_bwp_million",
    "gold_bwp_million",
    "iron_steel_products_bwp_million",
    "live_cattle_bwp_million",
    "machinery_electrical_bwp_million",
    "meat_products_bwp_million",
    "plastic_products_bwp_million",
    "salt_soda_ash_bwp_million",
    "textiles_bwp_million",
    "vehicles_transport_bwp_million",
    "other_goods_bwp_million",
    "total_goods_bwp_million"
]

# ------------------------------------------------------------
# Expand PDF rows containing multiple observations
#
# In the PDF, November and December 2025 occupy the same
# physical row. Each cell therefore contains two values
# separated by a line break. This logic separates them
# automatically before creating the DataFrame.
# ------------------------------------------------------------

expanded_rows = []

# Skip:
# row 0 = table title
# row 1 = reversed/rotated PDF headings

for row in raw_table_22[2:]:

    split_cells = []

    for value in row:

        if value is None:
            parts = [None]
        else:
            parts = [
                part.strip()
                for part in str(value).splitlines()
            ]

        split_cells.append(parts)

    number_of_lines = max(
        len(parts) for parts in split_cells
    )

    # Expand a multi-observation row only when the period cell
    # also contains the same number of separate labels.
    if (
        number_of_lines > 1
        and len(split_cells[0]) == number_of_lines
    ):

        for line_number in range(number_of_lines):

            reconstructed_row = []

            for parts in split_cells:

                if len(parts) == number_of_lines:
                    reconstructed_row.append(
                        parts[line_number]
                    )
                elif len(parts) == 1:
                    reconstructed_row.append(
                        parts[0]
                    )
                else:
                    reconstructed_row.append(None)

            expanded_rows.append(reconstructed_row)

    else:
        expanded_rows.append(row)

# ------------------------------------------------------------
# Create the cleaned table
# ------------------------------------------------------------

statsbots_exports_raw = pd.DataFrame(
    expanded_rows,
    columns=column_names
)

statsbots_exports_raw = (
    statsbots_exports_raw
    .dropna(how="all")
    .reset_index(drop=True)
)

statsbots_exports_raw["period"] = (
    statsbots_exports_raw["period"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
)

# ------------------------------------------------------------
# Convert PDF strings to numeric values
# ------------------------------------------------------------

numeric_columns = [
    column
    for column in statsbots_exports_raw.columns
    if column != "period"
]

def parse_pdf_number(value):

    if value is None:
        return np.nan

    value = str(value).strip()

    if value in {"", "None", "-", "."}:
        return np.nan

    # Parentheses denote negative values
    is_negative = (
        value.startswith("(")
        and value.endswith(")")
    )

    cleaned_value = (
        value
        .replace(",", "")
        .replace("(", "")
        .replace(")", "")
        .replace("%", "")
        .strip()
    )

    try:
        number = float(cleaned_value)

        return -number if is_negative else number

    except ValueError:
        return np.nan

for column in numeric_columns:
    statsbots_exports_raw[column] = (
        statsbots_exports_raw[column]
        .apply(parse_pdf_number)
    )

# ------------------------------------------------------------
# Validate the completed extraction
# ------------------------------------------------------------

assert statsbots_exports_raw.shape == (40, 15), (
    "Unexpected shape after cleaning: "
    f"{statsbots_exports_raw.shape}"
)

assert "Nov Dec" not in statsbots_exports_raw["period"].tolist(), (
    "November and December were not separated."
)

total_2024 = statsbots_exports_raw.loc[
    statsbots_exports_raw["period"] == "Total_2024",
    "total_goods_bwp_million"
].iloc[0]

total_2025 = statsbots_exports_raw.loc[
    statsbots_exports_raw["period"] == "Total_2025",
    "total_goods_bwp_million"
].iloc[0]

assert np.isclose(total_2024, 63344.5), (
    f"Unexpected 2024 total: {total_2024}"
)

assert np.isclose(total_2025, 74259.0), (
    f"Unexpected 2025 total: {total_2025}"
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nCleaned table shape:")
print(statsbots_exports_raw.shape)

print("\nOfficial annual totals:")
print(f"2024: P{total_2024:,.1f} million")
print(f"2025: P{total_2025:,.1f} million")

print("\nRows surrounding November and December 2025:")

display(
    statsbots_exports_raw.loc[
        29:34,
        [
            "period",
            "coal_bwp_million",
            "copper_bwp_million",
            "diamonds_bwp_million",
            "machinery_electrical_bwp_million",
            "other_goods_bwp_million",
            "total_goods_bwp_million"
        ]
    ]
)

print("\nMissing values in complete extracted table:")
print(statsbots_exports_raw.isna().sum())

Selected table:
TABLE 2.2: Principal Exports Commodity Groups – January 2024 to December 2025 (Million Pula)

Cleaned table shape:
(40, 15)

Official annual totals:
2024: P63,344.5 million
2025: P74,259.0 million

Rows surrounding November and December 2025:


,period,coal_bwp_million,copper_bwp_million,diamonds_bwp_million,machinery_electrical_bwp_million,other_goods_bwp_million,total_goods_bwp_million
29,Oct,32.3000,930.4000,"2,390.3000",339.6000,258.2000,"4,330.5000"
30,Nov,31.2000,817.8000,"7,135.4000",259.6000,163.5000,"8,762.6000"
31,Dec,22.6000,944.9000,"3,013.9000",129.7000,151.3000,"4,635.1000"
32,Q4,86.0000,"2,693.1000","12,539.6000",728.9000,573.0000,"17,728.3000"
33,Total_2025,352.4000,"11,025.6000","53,923.2000","3,066.1000","1,892.5000","74,259.0000"
34,Change,-8.6000,127.2000,"-4,121.5000",-129.8000,-12.3000,"-4,127.5000"



Missing values in complete extracted table:
period                              0
coal_bwp_million                    1
copper_bwp_million                  1
diamonds_bwp_million                1
gold_bwp_million                    1
iron_steel_products_bwp_million     1
live_cattle_bwp_million             1
machinery_electrical_bwp_million    1
meat_products_bwp_million           1
plastic_products_bwp_million        1
salt_soda_ash_bwp_million           1
textiles_bwp_million                1
vehicles_transport_bwp_million      2
other_goods_bwp_million             1
total_goods_bwp_million             1
dtype: int64


## Step 4 — Build monthly and annual export tables
Separates monthly rows from summary rows, totals each year, and reconciles the totals against the figures Statistics Botswana publishes.

In [7]:
# ============================================================
# STEP 4: CREATE MONTHLY AND ANNUAL EXPORT TABLES
# AND RECONCILE THEM TO THE PUBLISHED TOTALS
# ============================================================

# ------------------------------------------------------------
# Identify genuine monthly rows
# ------------------------------------------------------------

month_numbers = {
    "Jan_2024": 1,
    "Jan_2025": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12
}

monthly_records = []
current_year = None

for _, row in statsbots_exports_raw.iterrows():

    period = row["period"]

    if period == "Jan_2024":
        current_year = 2024

    elif period == "Jan_2025":
        current_year = 2025

    if period in month_numbers and current_year is not None:

        monthly_row = row.copy()

        monthly_row["reporting_year"] = current_year
        monthly_row["reporting_month"] = month_numbers[period]

        monthly_records.append(monthly_row)

statsbots_monthly_exports = pd.DataFrame(
    monthly_records
)

statsbots_monthly_exports["reporting_date"] = pd.to_datetime(
    {
        "year": statsbots_monthly_exports["reporting_year"],
        "month": statsbots_monthly_exports["reporting_month"],
        "day": 1
    }
)

statsbots_monthly_exports = (
    statsbots_monthly_exports
    .sort_values(["reporting_year", "reporting_month"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Extract official annual-total rows
# ------------------------------------------------------------

statsbots_annual_exports = (
    statsbots_exports_raw[
        statsbots_exports_raw["period"].isin(
            ["Total_2024", "Total_2025"]
        )
    ]
    .copy()
)

statsbots_annual_exports["reporting_year"] = (
    statsbots_annual_exports["period"]
    .str.extract(r"(\d{4})")[0]
    .astype(int)
)

statsbots_annual_exports = (
    statsbots_annual_exports
    .sort_values("reporting_year")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Confirm monthly and annual completeness
# ------------------------------------------------------------

assert len(statsbots_monthly_exports) == 24, (
    f"Expected 24 monthly rows; found "
    f"{len(statsbots_monthly_exports)}."
)

monthly_missing_values = (
    statsbots_monthly_exports[numeric_columns]
    .isna()
    .sum()
    .sum()
)

assert monthly_missing_values == 0, (
    f"Found {monthly_missing_values} missing monthly values."
)

assert len(statsbots_annual_exports) == 2, (
    f"Expected two annual-total rows; found "
    f"{len(statsbots_annual_exports)}."
)

# ------------------------------------------------------------
# Calculate annual totals from the monthly observations
# ------------------------------------------------------------

calculated_annual_totals = (
    statsbots_monthly_exports
    .groupby(
        "reporting_year",
        as_index=False
    )[numeric_columns]
    .sum()
)

# ------------------------------------------------------------
# Reshape calculated and published totals for comparison
# ------------------------------------------------------------

calculated_long = calculated_annual_totals.melt(
    id_vars="reporting_year",
    value_vars=numeric_columns,
    var_name="export_measure",
    value_name="calculated_from_months"
)

published_long = statsbots_annual_exports.melt(
    id_vars="reporting_year",
    value_vars=numeric_columns,
    var_name="export_measure",
    value_name="published_annual_total"
)

# Correct merge syntax:
# how describes the join method;
# validate checks the one-to-one relationship.

annual_reconciliation = calculated_long.merge(
    published_long,
    on=["reporting_year", "export_measure"],
    how="inner",
    validate="one_to_one"
)

annual_reconciliation["difference_bwp_million"] = (
    annual_reconciliation["calculated_from_months"]
    - annual_reconciliation["published_annual_total"]
)

annual_reconciliation["absolute_difference_bwp_million"] = (
    annual_reconciliation["difference_bwp_million"].abs()
)

maximum_reconciliation_difference = (
    annual_reconciliation[
        "absolute_difference_bwp_million"
    ].max()
)

# Monthly and annual figures are both rounded to one decimal,
# so differences of up to P0.2 million are expected.

assert maximum_reconciliation_difference <= 0.25, (
    "Monthly values do not reconcile sufficiently closely "
    "to the published annual totals. Maximum difference: "
    f"P{maximum_reconciliation_difference:.4f} million."
)

# ------------------------------------------------------------
# Display validation results
# ------------------------------------------------------------

print("Monthly observations:", len(statsbots_monthly_exports))
print("Annual observations:", len(statsbots_annual_exports))
print("Missing monthly values:", monthly_missing_values)

print(
    "Largest monthly-to-annual reconciliation difference:",
    f"P{maximum_reconciliation_difference:,.4f} million"
)

print("\nOfficial published annual export totals:")

display(
    statsbots_annual_exports[
        [
            "reporting_year",
            "coal_bwp_million",
            "copper_bwp_million",
            "diamonds_bwp_million",
            "machinery_electrical_bwp_million",
            "other_goods_bwp_million",
            "total_goods_bwp_million"
        ]
    ]
)

print("\nTotal-goods reconciliation:")

display(
    annual_reconciliation[
        annual_reconciliation["export_measure"]
        == "total_goods_bwp_million"
    ][
        [
            "reporting_year",
            "calculated_from_months",
            "published_annual_total",
            "difference_bwp_million"
        ]
    ]
)

Monthly observations: 24
Annual observations: 2
Missing monthly values: 0
Largest monthly-to-annual reconciliation difference: P0.2000 million

Official published annual export totals:


,reporting_year,coal_bwp_million,copper_bwp_million,diamonds_bwp_million,machinery_electrical_bwp_million,other_goods_bwp_million,total_goods_bwp_million
0,2024,428.8000,"10,064.3000","43,955.5000","2,928.6000","1,508.4000","63,344.5000"
1,2025,352.4000,"11,025.6000","53,923.2000","3,066.1000","1,892.5000","74,259.0000"



Total-goods reconciliation:


,reporting_year,calculated_from_months,published_annual_total,difference_bwp_million
26,2024,"63,344.4000","63,344.5000",-0.1000
27,2025,"74,258.9000","74,259.0000",-0.1000


## Step 5 — Extract Table 1.0: Total merchandise trade
Extracts total imports and exports for 2024–2025 and verifies the total exports figure.

In [10]:
# ============================================================
# STEP 5: EXTRACT TABLE 1.0 AND VERIFY TOTAL EXPORTS
# Total Merchandise Trade, 2024–2025
# ============================================================

# ------------------------------------------------------------
# Retrieve Table 1.0 preserved during Step 2
# ------------------------------------------------------------

table_10_key = "page_19_table_1"
raw_table_10 = all_pdf_tables[table_10_key]

table_10_title = str(raw_table_10[0][0])

assert "TABLE 1.0" in table_10_title.upper(), (
    f"Unexpected table selected: {table_10_title}"
)

print("Selected table:")
print(table_10_title)

trade_column_names = [
    "period",
    "imports_cif_bwp_million",
    "imports_fob_bwp_million",
    "freight_bwp_million",
    "insurance_bwp_million",
    "domestic_exports_bwp_million",
    "reexports_reported_bwp_million",
    "total_exports_bwp_million",
    "trade_balance_bwp_million"
]

# ------------------------------------------------------------
# Reusable function for splitting multi-line PDF rows
# ------------------------------------------------------------

def expand_multiline_pdf_rows(rows):

    expanded_rows = []

    for row in rows:

        split_cells = []

        for value in row:

            if value is None:
                parts = [None]
            else:
                parts = [
                    part.strip()
                    for part in str(value).splitlines()
                ]

            split_cells.append(parts)

        number_of_lines = max(
            len(parts) for parts in split_cells
        )

        if (
            number_of_lines > 1
            and len(split_cells[0]) == number_of_lines
        ):

            for line_number in range(number_of_lines):

                reconstructed_row = []

                for parts in split_cells:

                    if len(parts) == number_of_lines:
                        reconstructed_row.append(
                            parts[line_number]
                        )
                    elif len(parts) == 1:
                        reconstructed_row.append(
                            parts[0]
                        )
                    else:
                        reconstructed_row.append(None)

                expanded_rows.append(reconstructed_row)

        else:
            expanded_rows.append(row)

    return expanded_rows

# ------------------------------------------------------------
# Expand and clean Table 1.0
# ------------------------------------------------------------

trade_expanded_rows = expand_multiline_pdf_rows(
    raw_table_10[2:]
)

statsbots_trade_raw = pd.DataFrame(
    trade_expanded_rows,
    columns=trade_column_names
)

statsbots_trade_raw = (
    statsbots_trade_raw
    .dropna(how="all")
    .reset_index(drop=True)
)

statsbots_trade_raw["period"] = (
    statsbots_trade_raw["period"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
)

trade_numeric_columns = [
    column
    for column in statsbots_trade_raw.columns
    if column != "period"
]

for column in trade_numeric_columns:
    statsbots_trade_raw[column] = (
        statsbots_trade_raw[column]
        .apply(parse_pdf_number)
    )

# ------------------------------------------------------------
# Create the 24-row monthly trade table
# ------------------------------------------------------------

monthly_trade_records = []
current_year = None

for _, row in statsbots_trade_raw.iterrows():

    period = row["period"]

    if period == "Jan_2024":
        current_year = 2024

    elif period == "Jan_2025":
        current_year = 2025

    if period in month_numbers and current_year is not None:

        monthly_row = row.copy()

        monthly_row["reporting_year"] = current_year
        monthly_row["reporting_month"] = month_numbers[period]

        monthly_trade_records.append(monthly_row)

statsbots_monthly_trade = pd.DataFrame(
    monthly_trade_records
)

statsbots_monthly_trade = (
    statsbots_monthly_trade
    .sort_values(["reporting_year", "reporting_month"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Extract the official annual-total rows
# ------------------------------------------------------------

statsbots_annual_trade = (
    statsbots_trade_raw[
        statsbots_trade_raw["period"].isin(
            ["Total_2024", "Total_2025"]
        )
    ]
    .copy()
)

statsbots_annual_trade["reporting_year"] = (
    statsbots_annual_trade["period"]
    .str.extract(r"(\d{4})")[0]
    .astype(int)
)

statsbots_annual_trade = (
    statsbots_annual_trade
    .sort_values("reporting_year")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validate only the fields needed for this cross-check
# ------------------------------------------------------------

assert len(statsbots_monthly_trade) == 24, (
    f"Expected 24 monthly rows; found "
    f"{len(statsbots_monthly_trade)}."
)

assert len(statsbots_annual_trade) == 2, (
    f"Expected two annual rows; found "
    f"{len(statsbots_annual_trade)}."
)

assert (
    statsbots_monthly_trade[
        "total_exports_bwp_million"
    ].isna().sum()
    == 0
), "Missing monthly total-export values."

assert (
    statsbots_annual_trade[
        "total_exports_bwp_million"
    ].isna().sum()
    == 0
), "Missing annual total-export values."

# Preserve and report blank re-export cells, but do not modify them
blank_reexport_rows = statsbots_monthly_trade[
    statsbots_monthly_trade[
        "reexports_reported_bwp_million"
    ].isna()
][
    [
        "reporting_year",
        "reporting_month",
        "domestic_exports_bwp_million",
        "reexports_reported_bwp_million",
        "total_exports_bwp_million"
    ]
].copy()

# ------------------------------------------------------------
# Compare Table 1.0 and Table 2.2 monthly
# ------------------------------------------------------------

monthly_crosscheck = (
    statsbots_monthly_trade[
        [
            "reporting_year",
            "reporting_month",
            "total_exports_bwp_million"
        ]
    ]
    .merge(
        statsbots_monthly_exports[
            [
                "reporting_year",
                "reporting_month",
                "total_goods_bwp_million"
            ]
        ],
        on=["reporting_year", "reporting_month"],
        how="inner",
        validate="one_to_one"
    )
)

monthly_crosscheck["difference_bwp_million"] = (
    monthly_crosscheck["total_exports_bwp_million"]
    - monthly_crosscheck["total_goods_bwp_million"]
)

maximum_monthly_table_difference = (
    monthly_crosscheck["difference_bwp_million"]
    .abs()
    .max()
)

# ------------------------------------------------------------
# Compare Table 1.0 and Table 2.2 annually
# ------------------------------------------------------------

annual_crosscheck = (
    statsbots_annual_trade[
        [
            "reporting_year",
            "total_exports_bwp_million"
        ]
    ]
    .merge(
        statsbots_annual_exports[
            [
                "reporting_year",
                "total_goods_bwp_million"
            ]
        ],
        on="reporting_year",
        how="inner",
        validate="one_to_one"
    )
)

annual_crosscheck["difference_bwp_million"] = (
    annual_crosscheck["total_exports_bwp_million"]
    - annual_crosscheck["total_goods_bwp_million"]
)

maximum_annual_table_difference = (
    annual_crosscheck["difference_bwp_million"]
    .abs()
    .max()
)

assert maximum_monthly_table_difference <= 0.1, (
    "Tables 1.0 and 2.2 do not agree monthly."
)

assert maximum_annual_table_difference <= 0.1, (
    "Tables 1.0 and 2.2 do not agree annually."
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("Monthly trade observations:", len(statsbots_monthly_trade))

print(
    "Blank reported re-export cells:",
    len(blank_reexport_rows)
)

print(
    "Largest monthly difference between Tables 1.0 and 2.2:",
    f"P{maximum_monthly_table_difference:,.4f} million"
)

print(
    "Largest annual difference between Tables 1.0 and 2.2:",
    f"P{maximum_annual_table_difference:,.4f} million"
)

print("\nAnnual independent cross-check:")
display(annual_crosscheck)

print("\nOfficial annual trade measures:")
display(
    statsbots_annual_trade[
        [
            "reporting_year",
            "imports_cif_bwp_million",
            "domestic_exports_bwp_million",
            "reexports_reported_bwp_million",
            "total_exports_bwp_million",
            "trade_balance_bwp_million"
        ]
    ]
)

print("\nBlank re-export cells retained from the source:")
display(blank_reexport_rows)

Selected table:
TABLE 1.0: Total Merchandise Trade – January 2024 to December 2025 (Million Pula)
Monthly trade observations: 24
Blank reported re-export cells: 4
Largest monthly difference between Tables 1.0 and 2.2: P0.0000 million
Largest annual difference between Tables 1.0 and 2.2: P0.0000 million

Annual independent cross-check:


,reporting_year,total_exports_bwp_million,total_goods_bwp_million,difference_bwp_million
0,2024,"63,344.5000","63,344.5000",0.0000
1,2025,"74,259.0000","74,259.0000",0.0000



Official annual trade measures:


,reporting_year,imports_cif_bwp_million,domestic_exports_bwp_million,reexports_reported_bwp_million,total_exports_bwp_million,trade_balance_bwp_million
0,2024,"97,886.1000","63,260.3000",84.2000,"63,344.5000","-34,541.6000"
1,2025,"86,676.3000","74,239.7000",19.3000,"74,259.0000","-12,417.3000"



Blank re-export cells retained from the source:


,reporting_year,reporting_month,domestic_exports_bwp_million,reexports_reported_bwp_million,total_exports_bwp_million
11,2024,12,"5,156.5000",NaN,"5,156.5000"
14,2025,3,"5,905.2000",NaN,"5,905.2000"
17,2025,6,"7,578.9000",NaN,"7,578.9000"
18,2025,7,"7,503.4000",NaN,"7,503.4000"


## Step 6 — Extract Table 2.3: Diamond trade
Finds the diamond trade table by its content rather than only its page number, then extracts and validates it.

In [11]:
# ============================================================
# STEP 6: Extract and validate Table 2.3 - Diamonds Trade
# ============================================================

# ------------------------------------------------------------
# Locate Table 2.3 without relying only on its page number
# ------------------------------------------------------------
diamond_table_key = None

for table_key, table in all_pdf_tables.items():
    table_frame = pd.DataFrame(table)
    table_text = " ".join(
        table_frame.fillna("").astype(str).head(4).to_numpy().ravel()
    )

    if "TABLE 2.3: Diamonds Trade" in table_text:
        diamond_table_key = table_key
        break

assert diamond_table_key is not None, "Table 2.3 was not found in the PDF."

diamond_pdf_table = pd.DataFrame(all_pdf_tables[diamond_table_key])
diamond_table_title = str(diamond_pdf_table.iloc[0, 0]).strip()

# ------------------------------------------------------------
# Apply meaningful names to the import and export blocks
# ------------------------------------------------------------
diamond_columns = [
    "period",
    "imports_71021000_bwp_million",
    "imports_71022100_bwp_million",
    "imports_71023100_bwp_million",
    "imports_71023900_bwp_million",
    "diamond_imports_total_bwp_million",
    "exports_71021000_bwp_million",
    "exports_71022100_bwp_million",
    "exports_71023100_bwp_million",
    "exports_71023900_bwp_million",
    "diamond_exports_total_bwp_million"
]

# Rows 0-2 contain the title and multi-level headers.
statsbots_diamond_trade_raw = diamond_pdf_table.iloc[3:, :11].copy()
statsbots_diamond_trade_raw.columns = diamond_columns
statsbots_diamond_trade_raw = statsbots_diamond_trade_raw.reset_index(drop=True)

statsbots_diamond_trade_raw["period"] = (
    statsbots_diamond_trade_raw["period"].astype(str).str.strip()
)

diamond_numeric_columns = diamond_columns[1:]

for column in diamond_numeric_columns:
    statsbots_diamond_trade_raw[column] = pd.to_numeric(
        statsbots_diamond_trade_raw[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("(", "-", regex=False)
        .str.replace(")", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

# ------------------------------------------------------------
# Assign each row to 2024 or 2025
# ------------------------------------------------------------
current_year = None
diamond_year_values = []

for period in statsbots_diamond_trade_raw["period"]:
    if period == "Jan_2024":
        current_year = 2024
    elif period == "Jan_2025":
        current_year = 2025

    diamond_year_values.append(current_year)

statsbots_diamond_trade_raw.insert(
    1, "reporting_year", diamond_year_values
)

assert statsbots_diamond_trade_raw["reporting_year"].notna().all(), (
    "At least one Table 2.3 row could not be assigned to a year."
)

# ------------------------------------------------------------
# Create the 24 monthly observations
# ------------------------------------------------------------
diamond_month_numbers = {
    "Jan_2024": 1,
    "Jan_2025": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12
}

statsbots_monthly_diamond_trade = (
    statsbots_diamond_trade_raw[
        statsbots_diamond_trade_raw["period"].isin(diamond_month_numbers)
    ]
    .copy()
)

statsbots_monthly_diamond_trade.insert(
    2,
    "reporting_month",
    statsbots_monthly_diamond_trade["period"].map(diamond_month_numbers)
)

statsbots_monthly_diamond_trade = (
    statsbots_monthly_diamond_trade
    .sort_values(["reporting_year", "reporting_month"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Create the two official annual observations
# ------------------------------------------------------------
statsbots_annual_diamond_trade = (
    statsbots_diamond_trade_raw[
        statsbots_diamond_trade_raw["period"].isin(
            ["Total_2024", "Total_2025"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validate completeness and uniqueness
# ------------------------------------------------------------
assert len(statsbots_monthly_diamond_trade) == 24, (
    f"Expected 24 monthly diamond rows; found "
    f"{len(statsbots_monthly_diamond_trade)}."
)

assert len(statsbots_annual_diamond_trade) == 2, (
    f"Expected two annual diamond rows; found "
    f"{len(statsbots_annual_diamond_trade)}."
)

assert not statsbots_monthly_diamond_trade.duplicated(
    ["reporting_year", "reporting_month"]
).any(), "Duplicate monthly diamond observations were found."

assert (
    statsbots_monthly_diamond_trade[diamond_numeric_columns]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing values remain in the monthly diamond observations."

# ------------------------------------------------------------
# Check whether the detailed HS values add to the flow totals
# Small differences are expected because values are rounded
# to one decimal place in the report
# ------------------------------------------------------------
diamond_import_component_columns = [
    "imports_71021000_bwp_million",
    "imports_71022100_bwp_million",
    "imports_71023100_bwp_million",
    "imports_71023900_bwp_million"
]

diamond_export_component_columns = [
    "exports_71021000_bwp_million",
    "exports_71022100_bwp_million",
    "exports_71023100_bwp_million",
    "exports_71023900_bwp_million"
]

statsbots_diamond_trade_raw["calculated_diamond_imports_bwp_million"] = (
    statsbots_diamond_trade_raw[
        diamond_import_component_columns
    ].sum(axis=1)
)

statsbots_diamond_trade_raw[
    "import_component_difference_bwp_million"
] = (
    statsbots_diamond_trade_raw[
        "calculated_diamond_imports_bwp_million"
    ]
    - statsbots_diamond_trade_raw[
        "diamond_imports_total_bwp_million"
    ]
)

statsbots_diamond_trade_raw["calculated_diamond_exports_bwp_million"] = (
    statsbots_diamond_trade_raw[
        diamond_export_component_columns
    ].sum(axis=1)
)

statsbots_diamond_trade_raw[
    "export_component_difference_bwp_million"
] = (
    statsbots_diamond_trade_raw[
        "calculated_diamond_exports_bwp_million"
    ]
    - statsbots_diamond_trade_raw[
        "diamond_exports_total_bwp_million"
    ]
)

maximum_component_difference = max(
    statsbots_diamond_trade_raw[
        "import_component_difference_bwp_million"
    ].abs().max(),
    statsbots_diamond_trade_raw[
        "export_component_difference_bwp_million"
    ].abs().max()
)

assert maximum_component_difference <= 0.21, (
    "The detailed diamond HS values do not reconcile with their totals."
)

# ------------------------------------------------------------
# Reconcile the monthly values with each published annual total
# ------------------------------------------------------------
calculated_diamond_annual = (
    statsbots_monthly_diamond_trade
    .groupby(
        "reporting_year",
        as_index=False
    )[diamond_numeric_columns]
    .sum()
)

calculated_diamond_long = calculated_diamond_annual.melt(
    id_vars="reporting_year",
    value_vars=diamond_numeric_columns,
    var_name="diamond_measure",
    value_name="calculated_from_months"
)

published_diamond_long = statsbots_annual_diamond_trade.melt(
    id_vars="reporting_year",
    value_vars=diamond_numeric_columns,
    var_name="diamond_measure",
    value_name="published_annual_total"
)

diamond_annual_reconciliation = calculated_diamond_long.merge(
    published_diamond_long,
    on=["reporting_year", "diamond_measure"],
    how="inner",
    validate="one_to_one"
)

diamond_annual_reconciliation["difference_bwp_million"] = (
    diamond_annual_reconciliation["calculated_from_months"]
    - diamond_annual_reconciliation["published_annual_total"]
)

maximum_annual_diamond_difference = (
    diamond_annual_reconciliation[
        "difference_bwp_million"
    ].abs().max()
)

assert maximum_annual_diamond_difference <= 0.21, (
    "Monthly diamond values do not reconcile with the annual totals."
)

# ------------------------------------------------------------
# Independently cross-check Table 2.3 against Table 2.2
# ------------------------------------------------------------
monthly_diamond_crosscheck = (
    statsbots_monthly_diamond_trade[
        [
            "reporting_year",
            "reporting_month",
            "diamond_exports_total_bwp_million"
        ]
    ]
    .merge(
        statsbots_monthly_exports[
            [
                "reporting_year",
                "reporting_month",
                "diamonds_bwp_million"
            ]
        ],
        on=["reporting_year", "reporting_month"],
        how="inner",
        validate="one_to_one"
    )
)

monthly_diamond_crosscheck["difference_bwp_million"] = (
    monthly_diamond_crosscheck[
        "diamond_exports_total_bwp_million"
    ]
    - monthly_diamond_crosscheck["diamonds_bwp_million"]
)

annual_diamond_crosscheck = (
    statsbots_annual_diamond_trade[
        [
            "reporting_year",
            "diamond_exports_total_bwp_million"
        ]
    ]
    .merge(
        statsbots_annual_exports[
            ["reporting_year", "diamonds_bwp_million"]
        ],
        on="reporting_year",
        how="inner",
        validate="one_to_one"
    )
)

annual_diamond_crosscheck["difference_bwp_million"] = (
    annual_diamond_crosscheck[
        "diamond_exports_total_bwp_million"
    ]
    - annual_diamond_crosscheck["diamonds_bwp_million"]
)

maximum_monthly_table_difference = (
    monthly_diamond_crosscheck[
        "difference_bwp_million"
    ].abs().max()
)

maximum_annual_table_difference = (
    annual_diamond_crosscheck[
        "difference_bwp_million"
    ].abs().max()
)

assert maximum_monthly_table_difference <= 0.01, (
    "Monthly diamond exports differ between Tables 2.2 and 2.3."
)

assert maximum_annual_table_difference <= 0.01, (
    "Annual diamond exports differ between Tables 2.2 and 2.3."
)

# ------------------------------------------------------------
# Display the key validation results
# ------------------------------------------------------------
print("Selected table:")
print(diamond_table_title)
print()

print(
    "Monthly diamond observations:",
    len(statsbots_monthly_diamond_trade)
)

print(
    "Missing monthly diamond values:",
    int(
        statsbots_monthly_diamond_trade[
            diamond_numeric_columns
        ].isna().sum().sum()
    )
)

print(
    "Largest detailed-component reconciliation difference: "
    f"P{maximum_component_difference:,.4f} million"
)

print(
    "Largest monthly-to-annual reconciliation difference: "
    f"P{maximum_annual_diamond_difference:,.4f} million"
)

print(
    "Largest monthly difference between Tables 2.2 and 2.3: "
    f"P{maximum_monthly_table_difference:,.4f} million"
)

print(
    "Largest annual difference between Tables 2.2 and 2.3: "
    f"P{maximum_annual_table_difference:,.4f} million"
)

print("\nOfficial annual diamond trade:")

annual_diamond_output = statsbots_annual_diamond_trade[
    [
        "reporting_year",
        "diamond_imports_total_bwp_million",
        "diamond_exports_total_bwp_million",
        "exports_71021000_bwp_million",
        "exports_71022100_bwp_million",
        "exports_71023100_bwp_million",
        "exports_71023900_bwp_million"
    ]
]

display(annual_diamond_output)

print("\nAnnual independent cross-check:")

display(annual_diamond_crosscheck)

Selected table:
TABLE 2.3: Diamonds Trade – January 2024 to December 2025 (Million Pula)

Monthly diamond observations: 24
Missing monthly diamond values: 0
Largest detailed-component reconciliation difference: P0.2000 million
Largest monthly-to-annual reconciliation difference: P0.2000 million
Largest monthly difference between Tables 2.2 and 2.3: P0.0000 million
Largest annual difference between Tables 2.2 and 2.3: P0.0000 million

Official annual diamond trade:


,reporting_year,diamond_imports_total_bwp_million,diamond_exports_total_bwp_million,exports_71021000_bwp_million,exports_71022100_bwp_million,exports_71023100_bwp_million,exports_71023900_bwp_million
0,2024,"18,263.2000","43,955.5000",12.7000,207.7000,"35,345.5000","8,389.7000"
1,2025,"9,329.2000","53,923.2000",123.2000,189.7000,"45,298.0000","8,312.4000"



Annual independent cross-check:


,reporting_year,diamond_exports_total_bwp_million,diamonds_bwp_million,difference_bwp_million
0,2024,"43,955.5000","43,955.5000",0.0000
1,2025,"53,923.2000","53,923.2000",0.0000


## Step 7 — Build the annual actual-results table
Combines Tables 1.0, 2.2, and 2.3 into a single annual table of actual results, in pula.

In [12]:
# ============================================================
# STEP 7: Build the authoritative annual actual-results table
# ============================================================

# ------------------------------------------------------------
# Select and label values from Tables 1.0, 2.2, and 2.3
# ------------------------------------------------------------
annual_trade_summary = statsbots_annual_trade[
    [
        "reporting_year",
        "imports_cif_bwp_million",
        "domestic_exports_bwp_million",
        "reexports_reported_bwp_million",
        "total_exports_bwp_million",
        "trade_balance_bwp_million"
    ]
].copy()

annual_product_summary = statsbots_annual_exports[
    [
        "reporting_year",
        "total_goods_bwp_million",
        "diamonds_bwp_million"
    ]
].rename(
    columns={
        "total_goods_bwp_million":
            "table_2_2_total_exports_bwp_million",
        "diamonds_bwp_million":
            "table_2_2_diamond_exports_bwp_million"
    }
)

annual_diamond_summary = statsbots_annual_diamond_trade[
    [
        "reporting_year",
        "diamond_imports_total_bwp_million",
        "diamond_exports_total_bwp_million"
    ]
].rename(
    columns={
        "diamond_exports_total_bwp_million":
            "table_2_3_diamond_exports_bwp_million"
    }
)

# ------------------------------------------------------------
# Combine the three independently extracted source tables
# ------------------------------------------------------------
statsbots_annual_actuals_bwp = (
    annual_trade_summary
    .merge(
        annual_product_summary,
        on="reporting_year",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        annual_diamond_summary,
        on="reporting_year",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("reporting_year")
    .reset_index(drop=True)
)

assert statsbots_annual_actuals_bwp["reporting_year"].tolist() == [
    2024, 2025
], "The consolidated table should contain exactly 2024 and 2025."

# ------------------------------------------------------------
# Confirm that all three source tables still agree
# ------------------------------------------------------------
total_export_difference = (
    statsbots_annual_actuals_bwp["total_exports_bwp_million"]
    - statsbots_annual_actuals_bwp[
        "table_2_2_total_exports_bwp_million"
    ]
)

diamond_export_difference = (
    statsbots_annual_actuals_bwp[
        "table_2_2_diamond_exports_bwp_million"
    ]
    - statsbots_annual_actuals_bwp[
        "table_2_3_diamond_exports_bwp_million"
    ]
)

maximum_consolidation_difference = max(
    total_export_difference.abs().max(),
    diamond_export_difference.abs().max()
)

assert maximum_consolidation_difference <= 0.01, (
    "The official source tables do not agree during consolidation."
)

# Confirm the Table 1.0 export accounting identity.
export_accounting_difference = (
    statsbots_annual_actuals_bwp["domestic_exports_bwp_million"]
    + statsbots_annual_actuals_bwp["reexports_reported_bwp_million"]
    - statsbots_annual_actuals_bwp["total_exports_bwp_million"]
)

assert export_accounting_difference.abs().max() <= 0.01, (
    "Domestic exports plus re-exports do not equal total exports."
)

# ------------------------------------------------------------
# Create the authoritative export and diversification measures
# ------------------------------------------------------------
statsbots_annual_actuals_bwp["diamond_exports_bwp_million"] = (
    statsbots_annual_actuals_bwp[
        "table_2_3_diamond_exports_bwp_million"
    ]
)

statsbots_annual_actuals_bwp["non_diamond_exports_bwp_million"] = (
    statsbots_annual_actuals_bwp["total_exports_bwp_million"]
    - statsbots_annual_actuals_bwp["diamond_exports_bwp_million"]
)

statsbots_annual_actuals_bwp["diamond_export_share_pct"] = (
    100
    * statsbots_annual_actuals_bwp["diamond_exports_bwp_million"]
    / statsbots_annual_actuals_bwp["total_exports_bwp_million"]
)

statsbots_annual_actuals_bwp["non_diamond_export_share_pct"] = (
    100 - statsbots_annual_actuals_bwp["diamond_export_share_pct"]
)

statsbots_annual_actuals_bwp["export_import_coverage_pct"] = (
    100
    * statsbots_annual_actuals_bwp["total_exports_bwp_million"]
    / statsbots_annual_actuals_bwp["imports_cif_bwp_million"]
)

# ------------------------------------------------------------
# Calculate changes between 2024 and 2025
# The 2024 growth values remain missing because this report
# does not contain a comparable 2023 annual observation.
# ------------------------------------------------------------
growth_measures = {
    "total_exports_bwp_million": "total_exports_yoy_pct",
    "diamond_exports_bwp_million": "diamond_exports_yoy_pct",
    "non_diamond_exports_bwp_million":
        "non_diamond_exports_yoy_pct",
    "imports_cif_bwp_million": "imports_yoy_pct"
}

for value_column, growth_column in growth_measures.items():
    statsbots_annual_actuals_bwp[growth_column] = (
        statsbots_annual_actuals_bwp[value_column].pct_change() * 100
    )

statsbots_annual_actuals_bwp[
    "diamond_share_change_percentage_points"
] = (
    statsbots_annual_actuals_bwp["diamond_export_share_pct"].diff()
)

statsbots_annual_actuals_bwp[
    "trade_balance_change_bwp_million"
] = (
    statsbots_annual_actuals_bwp["trade_balance_bwp_million"].diff()
)

total_export_change = (
    statsbots_annual_actuals_bwp["total_exports_bwp_million"].diff()
)

diamond_export_change = (
    statsbots_annual_actuals_bwp["diamond_exports_bwp_million"].diff()
)

statsbots_annual_actuals_bwp[
    "diamond_contribution_to_export_change_pct"
] = np.where(
    total_export_change.notna() & total_export_change.ne(0),
    100 * diamond_export_change / total_export_change,
    np.nan
)

# ------------------------------------------------------------
# Add source documentation
# ------------------------------------------------------------
statsbots_annual_actuals_bwp["currency"] = "BWP"
statsbots_annual_actuals_bwp["unit"] = "million"

statsbots_annual_actuals_bwp[
    "source_organization"
] = "Statistics Botswana"

statsbots_annual_actuals_bwp["source_report"] = (
    "International Merchandise Trade Statistics Monthly "
    "Digest - December 2025"
)

statsbots_annual_actuals_bwp["source_tables"] = (
    "Table 1.0; Table 2.2; Table 2.3"
)

statsbots_annual_actuals_bwp["validation_status"] = (
    "Passed independent table cross-checks"
)

# ------------------------------------------------------------
# Retain the final analysis-ready columns
# ------------------------------------------------------------
statsbots_annual_actuals_bwp = statsbots_annual_actuals_bwp[
    [
        "reporting_year",
        "imports_cif_bwp_million",
        "domestic_exports_bwp_million",
        "reexports_reported_bwp_million",
        "total_exports_bwp_million",
        "diamond_imports_total_bwp_million",
        "diamond_exports_bwp_million",
        "non_diamond_exports_bwp_million",
        "trade_balance_bwp_million",
        "diamond_export_share_pct",
        "non_diamond_export_share_pct",
        "export_import_coverage_pct",
        "total_exports_yoy_pct",
        "diamond_exports_yoy_pct",
        "non_diamond_exports_yoy_pct",
        "imports_yoy_pct",
        "diamond_share_change_percentage_points",
        "trade_balance_change_bwp_million",
        "diamond_contribution_to_export_change_pct",
        "currency",
        "unit",
        "source_organization",
        "source_report",
        "source_tables",
        "validation_status"
    ]
].copy()

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------
required_actual_columns = [
    "total_exports_bwp_million",
    "diamond_exports_bwp_million",
    "non_diamond_exports_bwp_million",
    "diamond_export_share_pct",
    "trade_balance_bwp_million"
]

assert (
    statsbots_annual_actuals_bwp[required_actual_columns]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing values remain in the required annual actual measures."

# ------------------------------------------------------------
# Display the final results
# ------------------------------------------------------------
print("Authoritative annual actual-results table created.")
print("Rows:", len(statsbots_annual_actuals_bwp))
print(
    "Years:",
    statsbots_annual_actuals_bwp["reporting_year"].tolist()
)
print(
    "Largest source-table consolidation difference: "
    f"P{maximum_consolidation_difference:,.4f} million"
)

print("\nCore annual export and diversification measures:")

display(
    statsbots_annual_actuals_bwp[
        [
            "reporting_year",
            "total_exports_bwp_million",
            "diamond_exports_bwp_million",
            "non_diamond_exports_bwp_million",
            "diamond_export_share_pct",
            "non_diamond_export_share_pct",
            "trade_balance_bwp_million",
            "total_exports_yoy_pct",
            "diamond_exports_yoy_pct",
            "non_diamond_exports_yoy_pct",
            "diamond_contribution_to_export_change_pct"
        ]
    ]
)

Authoritative annual actual-results table created.
Rows: 2
Years: [2024, 2025]
Largest source-table consolidation difference: P0.0000 million

Core annual export and diversification measures:


,reporting_year,total_exports_bwp_million,diamond_exports_bwp_million,non_diamond_exports_bwp_million,diamond_export_share_pct,non_diamond_export_share_pct,trade_balance_bwp_million,total_exports_yoy_pct,diamond_exports_yoy_pct,non_diamond_exports_yoy_pct,diamond_contribution_to_export_change_pct
0,2024,"63,344.5000","43,955.5000","19,389.0000",69.3912,30.6088,"-34,541.6000",NaN,NaN,NaN,NaN
1,2025,"74,259.0000","53,923.2000","20,335.8000",72.6150,27.3850,"-12,417.3000",17.2304,22.6768,4.8832,91.3253


## Step 8 — Convert pula to US dollars
Downloads official daily exchange rates from the Bank of Botswana and converts the monthly export figures from pula (BWP) to US dollars, so they're comparable with the UN Comtrade-based forecasts.

In [13]:
# ============================================================
# STEP 8: Download official Bank of Botswana exchange rates
# and convert monthly export actuals from BWP to USD
# ============================================================

from io import BytesIO
from urllib.parse import urlencode
from urllib.request import Request, urlopen

# ------------------------------------------------------------
# Download official daily rates for 2024-2025
# The USD field reports USD received per BWP.
# ------------------------------------------------------------
exchange_rate_base_url = (
    "https://www.bankofbotswana.bw/export/exchange-rates.csv"
)

exchange_rate_parameters = {
    "_format": "csv",
    "field_exchange_date_value[min]": "2024-01-01",
    "field_exchange_date_value[max]": "2025-12-31"
}

exchange_rate_download_url = (
    f"{exchange_rate_base_url}?"
    f"{urlencode(exchange_rate_parameters)}"
)

exchange_rate_request = Request(
    exchange_rate_download_url,
    headers={"User-Agent": "Mozilla/5.0"}
)

with urlopen(exchange_rate_request, timeout=90) as response:
    exchange_rate_csv_bytes = response.read()

assert len(exchange_rate_csv_bytes) > 0, (
    "The Bank of Botswana exchange-rate download was empty."
)

bank_of_botswana_daily_rates = pd.read_csv(
    BytesIO(exchange_rate_csv_bytes),
    encoding="utf-8-sig"
)

bank_of_botswana_daily_rates.columns = (
    bank_of_botswana_daily_rates.columns
    .str.strip()
    .str.lower()
)

assert {"date", "usd"}.issubset(
    bank_of_botswana_daily_rates.columns
), (
    "The downloaded file does not contain the expected "
    "Date and USD fields."
)

# ------------------------------------------------------------
# Clean and validate the daily USD-per-BWP observations
# ------------------------------------------------------------
bank_of_botswana_daily_rates["date"] = pd.to_datetime(
    bank_of_botswana_daily_rates["date"],
    format="%d %b %Y",
    errors="coerce"
)

bank_of_botswana_daily_rates["usd"] = pd.to_numeric(
    bank_of_botswana_daily_rates["usd"],
    errors="coerce"
)

bank_of_botswana_daily_rates = (
    bank_of_botswana_daily_rates
    .rename(columns={"usd": "usd_per_bwp"})
    .loc[:, ["date", "usd_per_bwp"]]
    .dropna(subset=["date", "usd_per_bwp"])
    .sort_values("date")
    .reset_index(drop=True)
)

bank_of_botswana_daily_rates = (
    bank_of_botswana_daily_rates[
        bank_of_botswana_daily_rates["date"].between(
            "2024-01-01",
            "2025-12-31"
        )
    ]
    .copy()
)

assert not bank_of_botswana_daily_rates.duplicated("date").any(), (
    "Duplicate exchange-rate dates were downloaded."
)

assert (
    sorted(
        bank_of_botswana_daily_rates["date"]
        .dt.year
        .unique()
        .tolist()
    )
    == [2024, 2025]
), "The exchange-rate data do not contain both required years."

assert bank_of_botswana_daily_rates["usd_per_bwp"].between(
    0.01,
    0.20
).all(), (
    "At least one USD-per-BWP rate falls outside a plausible range."
)

# Retain the reciprocal for interpretation.
bank_of_botswana_daily_rates["bwp_per_usd"] = (
    1 / bank_of_botswana_daily_rates["usd_per_bwp"]
)

bank_of_botswana_daily_rates["reporting_year"] = (
    bank_of_botswana_daily_rates["date"].dt.year
)

bank_of_botswana_daily_rates["reporting_month"] = (
    bank_of_botswana_daily_rates["date"].dt.month
)

# ------------------------------------------------------------
# Calculate monthly-average official exchange rates
# ------------------------------------------------------------
bank_of_botswana_monthly_rates = (
    bank_of_botswana_daily_rates
    .groupby(
        ["reporting_year", "reporting_month"],
        as_index=False
    )
    .agg(
        average_usd_per_bwp=("usd_per_bwp", "mean"),
        average_bwp_per_usd=("bwp_per_usd", "mean"),
        exchange_rate_observations=("date", "count"),
        first_rate_date=("date", "min"),
        last_rate_date=("date", "max")
    )
)

assert len(bank_of_botswana_monthly_rates) == 24, (
    f"Expected 24 monthly exchange-rate observations; found "
    f"{len(bank_of_botswana_monthly_rates)}."
)

assert (
    bank_of_botswana_monthly_rates[
        "exchange_rate_observations"
    ] >= 15
).all(), (
    "At least one month contains too few daily "
    "exchange-rate observations."
)

# ------------------------------------------------------------
# Merge rates with the 24 monthly Table 2.2 observations
# ------------------------------------------------------------
monthly_export_bwp_columns = [
    column
    for column in statsbots_monthly_exports.columns
    if column.endswith("_bwp_million")
]

statsbots_monthly_actuals_usd = (
    statsbots_monthly_exports[
        [
            "reporting_year",
            "reporting_month",
            "period"
        ]
        + monthly_export_bwp_columns
    ]
    .merge(
        bank_of_botswana_monthly_rates,
        on=["reporting_year", "reporting_month"],
        how="inner",
        validate="one_to_one"
    )
    .sort_values(["reporting_year", "reporting_month"])
    .reset_index(drop=True)
)

assert len(statsbots_monthly_actuals_usd) == 24, (
    "The exchange-rate merge did not preserve all "
    "24 monthly export rows."
)

# ------------------------------------------------------------
# Convert all Table 2.2 export measures from BWP million
# to USD million using each month's average official rate
# ------------------------------------------------------------
monthly_export_usd_columns = []

for bwp_column in monthly_export_bwp_columns:
    usd_column = bwp_column.replace(
        "_bwp_million",
        "_usd_million"
    )

    statsbots_monthly_actuals_usd[usd_column] = (
        statsbots_monthly_actuals_usd[bwp_column]
        * statsbots_monthly_actuals_usd[
            "average_usd_per_bwp"
        ]
    )

    monthly_export_usd_columns.append(usd_column)

statsbots_monthly_actuals_usd[
    "non_diamond_exports_bwp_million"
] = (
    statsbots_monthly_actuals_usd["total_goods_bwp_million"]
    - statsbots_monthly_actuals_usd["diamonds_bwp_million"]
)

statsbots_monthly_actuals_usd[
    "non_diamond_exports_usd_million"
] = (
    statsbots_monthly_actuals_usd["total_goods_usd_million"]
    - statsbots_monthly_actuals_usd["diamonds_usd_million"]
)

assert (
    statsbots_monthly_actuals_usd[
        monthly_export_usd_columns
    ]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing values remain after converting monthly exports to USD."

# ------------------------------------------------------------
# Aggregate the converted monthly values into annual USD actuals
# ------------------------------------------------------------
annual_product_actuals_usd = (
    statsbots_monthly_actuals_usd
    .groupby("reporting_year", as_index=False)[
        monthly_export_usd_columns
        + ["non_diamond_exports_usd_million"]
    ]
    .sum()
)

annual_exchange_rate_summary = (
    bank_of_botswana_daily_rates
    .groupby("reporting_year", as_index=False)
    .agg(
        annual_average_usd_per_bwp=("usd_per_bwp", "mean"),
        annual_average_bwp_per_usd=("bwp_per_usd", "mean"),
        exchange_rate_observations=("date", "count")
    )
)

statsbots_annual_actuals_usd = (
    annual_product_actuals_usd
    .merge(
        annual_exchange_rate_summary,
        on="reporting_year",
        how="inner",
        validate="one_to_one"
    )
    .rename(
        columns={
            "total_goods_usd_million":
                "total_exports_usd_million",
            "diamonds_usd_million":
                "diamond_exports_usd_million"
        }
    )
)

# ------------------------------------------------------------
# Calculate USD-based shares and growth
# ------------------------------------------------------------
statsbots_annual_actuals_usd[
    "diamond_export_share_pct"
] = (
    100
    * statsbots_annual_actuals_usd[
        "diamond_exports_usd_million"
    ]
    / statsbots_annual_actuals_usd[
        "total_exports_usd_million"
    ]
)

statsbots_annual_actuals_usd[
    "non_diamond_export_share_pct"
] = (
    100
    - statsbots_annual_actuals_usd[
        "diamond_export_share_pct"
    ]
)

statsbots_annual_actuals_usd[
    "total_exports_usd_yoy_pct"
] = (
    statsbots_annual_actuals_usd[
        "total_exports_usd_million"
    ]
    .pct_change()
    * 100
)

statsbots_annual_actuals_usd[
    "diamond_exports_usd_yoy_pct"
] = (
    statsbots_annual_actuals_usd[
        "diamond_exports_usd_million"
    ]
    .pct_change()
    * 100
)

statsbots_annual_actuals_usd[
    "non_diamond_exports_usd_yoy_pct"
] = (
    statsbots_annual_actuals_usd[
        "non_diamond_exports_usd_million"
    ]
    .pct_change()
    * 100
)

# ------------------------------------------------------------
# Add source and methodological documentation
# ------------------------------------------------------------
statsbots_annual_actuals_usd["currency"] = "USD"
statsbots_annual_actuals_usd["unit"] = "million"

statsbots_annual_actuals_usd["exchange_rate_source"] = (
    "Bank of Botswana"
)

statsbots_annual_actuals_usd["conversion_method"] = (
    "Monthly BWP values multiplied by monthly average "
    "official USD-per-BWP rate"
)

assert statsbots_annual_actuals_usd[
    "reporting_year"
].tolist() == [2024, 2025], (
    "The annual USD table does not contain both required years."
)

required_annual_usd_columns = [
    "total_exports_usd_million",
    "diamond_exports_usd_million",
    "non_diamond_exports_usd_million"
]

assert (
    statsbots_annual_actuals_usd[
        required_annual_usd_columns
    ]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing values remain in the annual USD actuals."

# ------------------------------------------------------------
# Display the results
# ------------------------------------------------------------
print("Official Bank of Botswana exchange rates downloaded.")
print(
    "Daily observations:",
    len(bank_of_botswana_daily_rates)
)
print(
    "Monthly averages:",
    len(bank_of_botswana_monthly_rates)
)
print(
    "Date range:",
    bank_of_botswana_daily_rates["date"].min().date(),
    "to",
    bank_of_botswana_daily_rates["date"].max().date()
)

print("\nAnnual exchange-rate summary:")

display(annual_exchange_rate_summary)

print("\nAnnual exports converted to USD million:")

display(
    statsbots_annual_actuals_usd[
        [
            "reporting_year",
            "total_exports_usd_million",
            "diamond_exports_usd_million",
            "non_diamond_exports_usd_million",
            "diamond_export_share_pct",
            "total_exports_usd_yoy_pct",
            "diamond_exports_usd_yoy_pct",
            "non_diamond_exports_usd_yoy_pct",
            "annual_average_usd_per_bwp",
            "annual_average_bwp_per_usd",
            "exchange_rate_observations"
        ]
    ]
)

Official Bank of Botswana exchange rates downloaded.
Daily observations: 491
Monthly averages: 24
Date range: 2024-01-03 to 2025-12-31

Annual exchange-rate summary:


,reporting_year,annual_average_usd_per_bwp,annual_average_bwp_per_usd,exchange_rate_observations
0,2024,0.0737,13.5631,245
1,2025,0.0741,13.4987,246



Annual exports converted to USD million:


,reporting_year,total_exports_usd_million,diamond_exports_usd_million,non_diamond_exports_usd_million,diamond_export_share_pct,total_exports_usd_yoy_pct,diamond_exports_usd_yoy_pct,non_diamond_exports_usd_yoy_pct,annual_average_usd_per_bwp,annual_average_bwp_per_usd,exchange_rate_observations
0,2024,"4,664.9797","3,234.8918","1,430.0879",69.3442,NaN,NaN,NaN,0.0737,13.5631,245
1,2025,"5,503.5160","3,996.2880","1,507.2280",72.6134,17.9751,23.5370,5.3941,0.0741,13.4987,246


## Step 9 — Save the finalized actual-data files

In [14]:
# ============================================================
# STEP 9: Create and export the finalized actual-data files
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Create a dedicated output folder
# ------------------------------------------------------------
statsbots_output_folder = Path("statistics_botswana_outputs")
statsbots_output_folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Build one authoritative annual master table containing
# both official BWP values and converted USD values
# ------------------------------------------------------------
annual_usd_for_master = statsbots_annual_actuals_usd[
    [
        "reporting_year",
        "total_exports_usd_million",
        "diamond_exports_usd_million",
        "non_diamond_exports_usd_million",
        "total_exports_usd_yoy_pct",
        "diamond_exports_usd_yoy_pct",
        "non_diamond_exports_usd_yoy_pct",
        "annual_average_usd_per_bwp",
        "annual_average_bwp_per_usd",
        "exchange_rate_observations",
        "exchange_rate_source",
        "conversion_method"
    ]
].copy()

statsbots_annual_actuals_master = (
    statsbots_annual_actuals_bwp
    .merge(
        annual_usd_for_master,
        on="reporting_year",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("reporting_year")
    .reset_index(drop=True)
)

assert statsbots_annual_actuals_master[
    "reporting_year"
].tolist() == [2024, 2025], (
    "The annual master table does not contain 2024 and 2025."
)

annual_master_required_columns = [
    "total_exports_bwp_million",
    "diamond_exports_bwp_million",
    "non_diamond_exports_bwp_million",
    "total_exports_usd_million",
    "diamond_exports_usd_million",
    "non_diamond_exports_usd_million",
    "annual_average_usd_per_bwp"
]

assert (
    statsbots_annual_actuals_master[
        annual_master_required_columns
    ]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing values remain in the required annual master fields."

# ------------------------------------------------------------
# Define the finalized file paths
# ------------------------------------------------------------
annual_master_path = (
    statsbots_output_folder
    / "statsbots_annual_actuals_2024_2025.csv"
)

monthly_bwp_path = (
    statsbots_output_folder
    / "statsbots_monthly_exports_bwp_2024_2025.csv"
)

monthly_usd_path = (
    statsbots_output_folder
    / "statsbots_monthly_exports_usd_2024_2025.csv"
)

monthly_rates_path = (
    statsbots_output_folder
    / "bank_of_botswana_monthly_exchange_rates_2024_2025.csv"
)

daily_rates_path = (
    statsbots_output_folder
    / "bank_of_botswana_daily_exchange_rates_2024_2025.csv"
)

# ------------------------------------------------------------
# Export the finalized datasets
# ------------------------------------------------------------
statsbots_annual_actuals_master.to_csv(
    annual_master_path,
    index=False
)

statsbots_monthly_exports.to_csv(
    monthly_bwp_path,
    index=False
)

statsbots_monthly_actuals_usd.to_csv(
    monthly_usd_path,
    index=False
)

bank_of_botswana_monthly_rates.to_csv(
    monthly_rates_path,
    index=False
)

bank_of_botswana_daily_rates.to_csv(
    daily_rates_path,
    index=False
)

# ------------------------------------------------------------
# Reload every file to verify the exports
# ------------------------------------------------------------
exported_files = {
    "annual_actuals_master": annual_master_path,
    "monthly_exports_bwp": monthly_bwp_path,
    "monthly_exports_usd": monthly_usd_path,
    "monthly_exchange_rates": monthly_rates_path,
    "daily_exchange_rates": daily_rates_path
}

expected_row_counts = {
    "annual_actuals_master": 2,
    "monthly_exports_bwp": 24,
    "monthly_exports_usd": 24,
    "monthly_exchange_rates": 24,
    "daily_exchange_rates": len(
        bank_of_botswana_daily_rates
    )
}

export_verification_records = []

for dataset_name, file_path in exported_files.items():
    reloaded_data = pd.read_csv(file_path)

    assert len(reloaded_data) == expected_row_counts[dataset_name], (
        f"{dataset_name} was expected to contain "
        f"{expected_row_counts[dataset_name]} rows but contained "
        f"{len(reloaded_data)} rows after export."
    )

    assert not any(
        str(column).startswith("Unnamed:")
        for column in reloaded_data.columns
    ), f"An unwanted index column was found in {dataset_name}."

    export_verification_records.append(
        {
            "dataset": dataset_name,
            "file_name": file_path.name,
            "rows": len(reloaded_data),
            "columns": len(reloaded_data.columns),
            "file_size_kb": (
                file_path.stat().st_size / 1024
            ),
            "verification_status": "Passed"
        }
    )

statsbots_export_verification = pd.DataFrame(
    export_verification_records
)

# ------------------------------------------------------------
# Display the final export summary
# ------------------------------------------------------------
print("Statistics Botswana actual-data exports completed.")
print(
    "Output folder:",
    statsbots_output_folder.resolve()
)
print("Files created:", len(exported_files))

display(statsbots_export_verification)

print("\nFinal annual master table:")

display(
    statsbots_annual_actuals_master[
        [
            "reporting_year",
            "total_exports_bwp_million",
            "diamond_exports_bwp_million",
            "non_diamond_exports_bwp_million",
            "total_exports_usd_million",
            "diamond_exports_usd_million",
            "non_diamond_exports_usd_million",
            "diamond_export_share_pct",
            "total_exports_usd_yoy_pct",
            "diamond_exports_usd_yoy_pct",
            "non_diamond_exports_usd_yoy_pct",
            "annual_average_bwp_per_usd",
            "validation_status"
        ]
    ]
)

Statistics Botswana actual-data exports completed.
Output folder: C:\Users\boemo\OneDrive\Desktop\School Documents\DSS 680\New Project Data\statistics_botswana_outputs
Files created: 5


,dataset,file_name,rows,columns,file_size_kb,verification_status
0,annual_actuals_master,statsbots_annual_actuals_2024_2025.csv,2,36,2.0273,Passed
1,monthly_exports_bwp,statsbots_monthly_exports_bwp_2024_2025.csv,24,18,2.7334,Passed
2,monthly_exports_usd,statsbots_monthly_exports_usd_2024_2025.csv,24,38,10.1455,Passed
3,monthly_exchange_rates,bank_of_botswana_monthly_exchange_rates_2024_2...,24,7,1.7324,Passed
4,daily_exchange_rates,bank_of_botswana_daily_exchange_rates_2024_202...,491,5,21.5635,Passed



Final annual master table:


,reporting_year,total_exports_bwp_million,diamond_exports_bwp_million,non_diamond_exports_bwp_million,total_exports_usd_million,diamond_exports_usd_million,non_diamond_exports_usd_million,diamond_export_share_pct,total_exports_usd_yoy_pct,diamond_exports_usd_yoy_pct,non_diamond_exports_usd_yoy_pct,annual_average_bwp_per_usd,validation_status
0,2024,"63,344.5000","43,955.5000","19,389.0000","4,664.9797","3,234.8918","1,430.0879",69.3912,NaN,NaN,NaN,13.5631,Passed independent table cross-checks
1,2025,"74,259.0000","53,923.2000","20,335.8000","5,503.5160","3,996.2880","1,507.2280",72.6150,17.9751,23.5370,5.3941,13.4987,Passed independent table cross-checks


## Step 10 — Evaluate the SAS LASSO 2024 forecast
Compares the LASSO model's 2024 forecasts with the official 2024 actual figures for total, diamond, and non-diamond exports.

In [15]:
# ============================================================
# STEP 10: Evaluate the SAS LASSO 2024 forecast against
# official Statistics Botswana 2024 actual results
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Locate the SAS LASSO score file
# ------------------------------------------------------------
lasso_score_path = Path("table_LASSOSCOREDATA.csv")

assert lasso_score_path.exists(), (
    "table_LASSOSCOREDATA.csv was not found. Place the SAS "
    "LASSO score CSV in the same folder as this notebook "
    "and rerun the cell."
)

assert "statsbots_annual_actuals_master" in globals(), (
    "Run Steps 1-9 before running the LASSO validation cell."
)

# ------------------------------------------------------------
# Import and validate the SAS LASSO score output
# ------------------------------------------------------------
lasso_scored_2024 = pd.read_csv(
    lasso_score_path,
    dtype={
        "product_year_id": "string",
        "product_category_code": "string"
    }
)

required_lasso_columns = [
    "product_year_id",
    "reporting_year",
    "product_category_code",
    "product_category_description",
    "annual_export_value_usd",
    "total_exports_all_products_usd",
    "is_diamond_category",
    "P_next_year_log_export_value",
    "EM_PREDICTION"
]

missing_lasso_columns = [
    column
    for column in required_lasso_columns
    if column not in lasso_scored_2024.columns
]

assert not missing_lasso_columns, (
    f"Required LASSO columns are missing: "
    f"{missing_lasso_columns}"
)

lasso_scored_2024["product_category_code"] = (
    lasso_scored_2024["product_category_code"]
    .str.strip()
    .str.zfill(2)
)

assert len(lasso_scored_2024) == 98, (
    f"Expected 98 scored product categories; found "
    f"{len(lasso_scored_2024)}."
)

assert (
    lasso_scored_2024["product_category_code"].nunique()
    == 98
), (
    "The LASSO score file does not contain "
    "98 unique product codes."
)

assert lasso_scored_2024["reporting_year"].eq(2023).all(), (
    "The SAS score rows are not all based on reporting year 2023."
)

assert (
    lasso_scored_2024[
        [
            "P_next_year_log_export_value",
            "EM_PREDICTION"
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing SAS LASSO predictions were found."

maximum_sas_prediction_difference = (
    lasso_scored_2024["P_next_year_log_export_value"]
    - lasso_scored_2024["EM_PREDICTION"]
).abs().max()

assert maximum_sas_prediction_difference <= 1e-10, (
    "The two SAS prediction columns do not agree."
)

# ------------------------------------------------------------
# Reverse the log1p target transformation
# ------------------------------------------------------------
lasso_scored_2024["target_year"] = 2024

lasso_scored_2024[
    "predicted_2024_export_value_usd"
] = np.expm1(
    lasso_scored_2024[
        "P_next_year_log_export_value"
    ]
)

assert np.isfinite(
    lasso_scored_2024[
        "predicted_2024_export_value_usd"
    ]
).all(), "Non-finite dollar predictions were produced."

assert (
    lasso_scored_2024[
        "predicted_2024_export_value_usd"
    ] >= 0
).all(), "At least one reversed LASSO prediction is negative."

# ------------------------------------------------------------
# Aggregate the 98 predictions
# ------------------------------------------------------------
predicted_2024_total_usd_million = (
    lasso_scored_2024[
        "predicted_2024_export_value_usd"
    ].sum()
    / 1_000_000
)

# The modeling dataset isolates HS 7102 diamonds from
# 71_OTHER, making Table 2.3 directly comparable.
diamond_score_rows = lasso_scored_2024.loc[
    lasso_scored_2024["is_diamond_category"].eq(1)
]

assert len(diamond_score_rows) == 1, (
    "Exactly one modeled diamond category was expected."
)

assert (
    diamond_score_rows[
        "product_category_code"
    ].iloc[0]
    == "7102"
), "The modeled diamond category is not HS 7102."

predicted_2024_diamond_usd_million = (
    diamond_score_rows[
        "predicted_2024_export_value_usd"
    ].sum()
    / 1_000_000
)

predicted_2024_non_diamond_usd_million = (
    predicted_2024_total_usd_million
    - predicted_2024_diamond_usd_million
)

# ------------------------------------------------------------
# Retain the product-level predictions for later analysis
# ------------------------------------------------------------
lasso_2024_product_predictions = (
    lasso_scored_2024[
        [
            "product_year_id",
            "reporting_year",
            "target_year",
            "product_category_code",
            "product_category_description",
            "is_diamond_category",
            "annual_export_value_usd",
            "P_next_year_log_export_value",
            "predicted_2024_export_value_usd"
        ]
    ]
    .sort_values(
        "predicted_2024_export_value_usd",
        ascending=False
    )
    .reset_index(drop=True)
)

lasso_2024_product_predictions[
    "predicted_2024_export_share_pct"
] = (
    100
    * lasso_2024_product_predictions[
        "predicted_2024_export_value_usd"
    ]
    / lasso_2024_product_predictions[
        "predicted_2024_export_value_usd"
    ].sum()
)

# ------------------------------------------------------------
# Retrieve official Statistics Botswana 2024 actuals
# ------------------------------------------------------------
actual_2024_row = statsbots_annual_actuals_master.loc[
    statsbots_annual_actuals_master[
        "reporting_year"
    ].eq(2024)
]

assert len(actual_2024_row) == 1, (
    "Exactly one official 2024 actual row was expected."
)

actual_2024_total_usd_million = float(
    actual_2024_row[
        "total_exports_usd_million"
    ].iloc[0]
)

actual_2024_diamond_usd_million = float(
    actual_2024_row[
        "diamond_exports_usd_million"
    ].iloc[0]
)

actual_2024_non_diamond_usd_million = float(
    actual_2024_row[
        "non_diamond_exports_usd_million"
    ].iloc[0]
)

# ------------------------------------------------------------
# Retrieve the 2023 UN Comtrade baseline
# ------------------------------------------------------------
assert (
    lasso_scored_2024[
        "total_exports_all_products_usd"
    ].nunique()
    == 1
), (
    "The 2023 total-export baseline is not "
    "consistent across score rows."
)

baseline_2023_total_usd_million = (
    lasso_scored_2024[
        "total_exports_all_products_usd"
    ].median()
    / 1_000_000
)

baseline_2023_diamond_usd_million = (
    diamond_score_rows[
        "annual_export_value_usd"
    ].sum()
    / 1_000_000
)

baseline_2023_non_diamond_usd_million = (
    baseline_2023_total_usd_million
    - baseline_2023_diamond_usd_million
)

# ------------------------------------------------------------
# Build the total, diamond, and non-diamond comparison
# ------------------------------------------------------------
lasso_2024_external_validation = pd.DataFrame(
    {
        "export_component": [
            "Total exports",
            "Diamond exports (HS 7102)",
            "Non-diamond exports"
        ],
        "baseline_2023_usd_million": [
            baseline_2023_total_usd_million,
            baseline_2023_diamond_usd_million,
            baseline_2023_non_diamond_usd_million
        ],
        "predicted_2024_usd_million": [
            predicted_2024_total_usd_million,
            predicted_2024_diamond_usd_million,
            predicted_2024_non_diamond_usd_million
        ],
        "actual_2024_usd_million": [
            actual_2024_total_usd_million,
            actual_2024_diamond_usd_million,
            actual_2024_non_diamond_usd_million
        ]
    }
)

# ------------------------------------------------------------
# Calculate forecast errors
# ------------------------------------------------------------
lasso_2024_external_validation[
    "forecast_error_usd_million"
] = (
    lasso_2024_external_validation[
        "predicted_2024_usd_million"
    ]
    - lasso_2024_external_validation[
        "actual_2024_usd_million"
    ]
)

lasso_2024_external_validation[
    "absolute_error_usd_million"
] = (
    lasso_2024_external_validation[
        "forecast_error_usd_million"
    ].abs()
)

lasso_2024_external_validation[
    "percentage_error_pct"
] = (
    100
    * lasso_2024_external_validation[
        "forecast_error_usd_million"
    ]
    / lasso_2024_external_validation[
        "actual_2024_usd_million"
    ]
)

lasso_2024_external_validation[
    "absolute_percentage_error_pct"
] = (
    lasso_2024_external_validation[
        "percentage_error_pct"
    ].abs()
)

# ------------------------------------------------------------
# Compare predicted and actual direction from 2023
# ------------------------------------------------------------
lasso_2024_external_validation[
    "predicted_change_from_2023_pct"
] = (
    100
    * (
        lasso_2024_external_validation[
            "predicted_2024_usd_million"
        ]
        / lasso_2024_external_validation[
            "baseline_2023_usd_million"
        ]
        - 1
    )
)

lasso_2024_external_validation[
    "actual_change_from_2023_pct"
] = (
    100
    * (
        lasso_2024_external_validation[
            "actual_2024_usd_million"
        ]
        / lasso_2024_external_validation[
            "baseline_2023_usd_million"
        ]
        - 1
    )
)

lasso_2024_external_validation[
    "direction_correct"
] = (
    np.sign(
        lasso_2024_external_validation[
            "predicted_change_from_2023_pct"
        ]
    )
    == np.sign(
        lasso_2024_external_validation[
            "actual_change_from_2023_pct"
        ]
    )
)

# ------------------------------------------------------------
# Add documentation fields
# ------------------------------------------------------------
lasso_2024_external_validation.insert(
    0,
    "model",
    "SAS LASSO Regression"
)

lasso_2024_external_validation.insert(
    1,
    "forecast_year",
    2024
)

lasso_2024_external_validation[
    "actual_source"
] = "Statistics Botswana"

lasso_2024_external_validation[
    "actual_currency_conversion"
] = "Monthly official Bank of Botswana rates"

lasso_2024_external_validation[
    "validation_scope"
] = (
    "External validation of total, HS 7102 diamonds, "
    "and residual non-diamond exports"
)

# ------------------------------------------------------------
# Display the results
# ------------------------------------------------------------
print("SAS LASSO score file validated.")
print(
    "Scored product categories:",
    len(lasso_scored_2024)
)
print(
    "Maximum difference between SAS prediction columns:",
    f"{maximum_sas_prediction_difference:.12f}"
)

print("\n2024 external-validation results:")

display(lasso_2024_external_validation)

print("\nTen largest predicted 2024 product categories:")

display(
    lasso_2024_product_predictions[
        [
            "product_category_code",
            "product_category_description",
            "predicted_2024_export_value_usd",
            "predicted_2024_export_share_pct"
        ]
    ].head(10)
)

SAS LASSO score file validated.
Scored product categories: 98
Maximum difference between SAS prediction columns: 0.000000000000

2024 external-validation results:


,model,forecast_year,export_component,baseline_2023_usd_million,predicted_2024_usd_million,actual_2024_usd_million,forecast_error_usd_million,absolute_error_usd_million,percentage_error_pct,absolute_percentage_error_pct,predicted_change_from_2023_pct,actual_change_from_2023_pct,direction_correct,actual_source,actual_currency_conversion,validation_scope
0,SAS LASSO Regression,2024,Total exports,"5,502.7165","4,803.3966","4,664.9797",138.4169,138.4169,2.9671,2.9671,-12.7086,-15.2241,True,Statistics Botswana,Monthly official Bank of Botswana rates,"External validation of total, HS 7102 diamonds..."
1,SAS LASSO Regression,2024,Diamond exports (HS 7102),"4,333.5664","4,086.8527","3,234.8918",851.9610,851.9610,26.3366,26.3366,-5.6931,-25.3527,True,Statistics Botswana,Monthly official Bank of Botswana rates,"External validation of total, HS 7102 diamonds..."
2,SAS LASSO Regression,2024,Non-diamond exports,"1,169.1501",716.5439,"1,430.0879",-713.5440,713.5440,-49.8951,49.8951,-38.7124,22.3186,False,Statistics Botswana,Monthly official Bank of Botswana rates,"External validation of total, HS 7102 diamonds..."



Ten largest predicted 2024 product categories:


,product_category_code,product_category_description,predicted_2024_export_value_usd,predicted_2024_export_share_pct
0,7102,"Diamonds, whether or not worked, but not mount...","4,086,852,736.7089",85.0826
1,26,"Ores, slag and ash","154,898,026.7658",3.2248
2,85,Electrical machinery and equipment and parts t...,"146,368,816.8533",3.0472
3,27,"Mineral fuels, mineral oils and products of th...","44,396,501.1103",0.9243
4,84,"Machinery and mechanical appliances, boilers, ...","43,111,512.4150",0.8975
5,28,Inorganic chemicals; organic and inorganic com...,"37,065,865.5130",0.7717
6,01,Animals; live,"36,159,925.0714",0.7528
7,87,Vehicles; other than railway or tramway rollin...,"30,978,631.6674",0.6449
8,39,Plastics and articles thereof,"29,608,445.0465",0.6164
9,25,"Salt; sulphur; earths, stone; plastering mater...","21,891,729.8998",0.4558


## Step 11 — Extend the LASSO forecast to 2025
Trains a direct two-year-ahead LASSO model in Python to forecast 2025 and compares it with the official 2025 figures.

In [16]:
# ============================================================
# STEP 11: Direct-horizon LASSO forecast for 2025
# Run after Step 10. This creates no new files.
# ============================================================

from pathlib import Path
from sklearn.linear_model import Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

master_path = Path("botswana_complete_master_2000_2023.csv")

assert master_path.exists(), (
    "Place botswana_complete_master_2000_2023.csv "
    "in the same folder as the notebook."
)

assert "lasso_scored_2024" in globals(), "Run Step 10 first."
assert "statsbots_annual_actuals_master" in globals(), (
    "Run Steps 1-10 first."
)

# ------------------------------------------------------------
# Import and validate the complete historical master data
# ------------------------------------------------------------

master = pd.read_csv(
    master_path,
    dtype={
        "product_category_code": "string",
        "product_year_id": "string"
    }
)

master["product_category_code"] = (
    master["product_category_code"]
    .str.strip()
    .str.zfill(2)
)

master = (
    master
    .sort_values(["product_category_code", "reporting_year"])
    .reset_index(drop=True)
)

assert len(master) == 2352, (
    f"Expected 2,352 observations; found {len(master)}."
)

assert master["product_category_code"].nunique() == 98, (
    "Expected 98 product categories."
)

# ------------------------------------------------------------
# Recreate the predictor transformations used for SAS modeling
# ------------------------------------------------------------

master["annual_export_value_usd_filled"] = (
    master["annual_export_value_usd"].fillna(0)
)

log_columns = {
    "annual_export_value_usd_filled":
        "LOG_annual_export_value_usd_filled",

    "annual_import_value_usd":
        "LOG_annual_import_value_usd",

    "export_avg_3yr_usd":
        "LOG_export_avg_3yr_usd",

    "export_avg_5yr_usd":
        "LOG_export_avg_5yr_usd",

    "export_value_lag1_usd":
        "LOG_export_value_lag1_usd",

    "export_value_lag2_usd":
        "LOG_export_value_lag2_usd",

    "export_value_lag3_usd":
        "LOG_export_value_lag3_usd",

    "export_value_lag4_usd":
        "LOG_export_value_lag4_usd",

    "export_value_lag5_usd":
        "LOG_export_value_lag5_usd",

    "total_exports_all_products_usd":
        "LOG_total_exports_all_products_usd",

    "total_imports_all_products_usd":
        "LOG_total_imports_all_products_usd"
}

for source, destination in log_columns.items():
    master[destination] = np.log1p(master[source])

predictors = [
    "LOG_annual_export_value_usd_filled",
    "LOG_annual_import_value_usd",
    "LOG_export_avg_3yr_usd",
    "LOG_export_avg_5yr_usd",
    "LOG_export_value_lag1_usd",
    "LOG_export_value_lag2_usd",
    "LOG_export_value_lag3_usd",
    "LOG_export_value_lag4_usd",
    "LOG_export_value_lag5_usd",
    "LOG_total_exports_all_products_usd",
    "LOG_total_imports_all_products_usd",
    "export_share_pct",
    "import_share_pct",
    "export_log_growth_1yr",
    "export_log_trend_5yr",
    "export_log_volatility_5yr",
    "normalized_trade_balance",
    "product_trade_balance_usd",
    "recorded_export_years_5yr",
    "is_diamond_category",
    "is_net_exporter",
    "no_recorded_exports"
]

missing_predictors = [
    column for column in predictors
    if column not in master.columns
]

assert not missing_predictors, (
    f"Missing required predictors: {missing_predictors}"
)

# ------------------------------------------------------------
# Function for time-based tuning and direct-horizon forecasting
# ------------------------------------------------------------

def fit_direct_lasso(horizon):
    """
    Fit a direct-horizon LASSO.

    horizon=1 predicts the next year.
    horizon=2 predicts two years ahead directly.
    """

    target = f"log_export_lead_{horizon}"

    master[target] = np.log1p(
        master
        .groupby("product_category_code")[
            "annual_export_value_usd_filled"
        ]
        .shift(-horizon)
    )

    eligible = master.loc[
        master["reporting_year"].between(
            2005,
            2023 - horizon
        )
        & master[predictors].notna().all(axis=1)
        & master[target].notna()
    ].copy()

    origin_years = sorted(
        eligible["reporting_year"].unique()
    )

    # Use the most recent approximately 22% of years
    # for time-based validation.
    number_validation_years = max(
        2,
        round(len(origin_years) * 0.22)
    )

    validation_years = origin_years[
        -number_validation_years:
    ]

    training_years = origin_years[
        :-number_validation_years
    ]

    train = eligible.loc[
        eligible["reporting_year"].isin(training_years)
    ]

    validate = eligible.loc[
        eligible["reporting_year"].isin(validation_years)
    ]

    # Tune the LASSO penalty using validation ASE.
    alpha_results = []

    for alpha in np.logspace(-4, 1, 60):

        candidate = make_pipeline(
            StandardScaler(),
            Lasso(
                alpha=alpha,
                max_iter=100000,
                tol=1e-7
            )
        )

        candidate.fit(
            train[predictors],
            train[target]
        )

        validation_prediction = candidate.predict(
            validate[predictors]
        )

        validation_ase = mean_squared_error(
            validate[target],
            validation_prediction
        )

        alpha_results.append(
            (alpha, validation_ase)
        )

    selected_alpha, validation_ase = min(
        alpha_results,
        key=lambda row: row[1]
    )

    # Refit using all eligible historical observations
    # after the penalty has been selected.
    final_model = make_pipeline(
        StandardScaler(),
        Lasso(
            alpha=selected_alpha,
            max_iter=100000,
            tol=1e-7
        )
    )

    final_model.fit(
        eligible[predictors],
        eligible[target]
    )

    # Score the 98 product categories using their 2023 data.
    score = master.loc[
        master["reporting_year"].eq(2023)
        & master[predictors].notna().all(axis=1)
    ].copy()

    assert len(score) == 98, (
        f"Expected 98 scoring observations; found {len(score)}."
    )

    score[f"predicted_log_h{horizon}"] = (
        final_model.predict(score[predictors])
    )

    score[f"predicted_usd_h{horizon}"] = (
        np.expm1(score[f"predicted_log_h{horizon}"])
        .clip(lower=0)
    )

    summary = {
        "horizon_years": horizon,
        "forecast_year": 2023 + horizon,
        "selected_alpha": selected_alpha,
        "validation_ase": validation_ase,
        "training_origin_years":
            f"{min(training_years)}-{max(training_years)}",
        "validation_origin_years":
            f"{min(validation_years)}-{max(validation_years)}",
        "refit_origin_years":
            f"{min(origin_years)}-{max(origin_years)}",
        "refit_rows": len(eligible)
    }

    return final_model, score, summary

# ------------------------------------------------------------
# Fit the one-year diagnostic and two-year forecast models
# ------------------------------------------------------------

lasso_h1_model, lasso_h1_score, h1_summary = (
    fit_direct_lasso(1)
)

lasso_h2_model, lasso_h2_score, h2_summary = (
    fit_direct_lasso(2)
)

lasso_horizon_validation = pd.DataFrame(
    [h1_summary, h2_summary]
)

# ------------------------------------------------------------
# Compare the Python h=1 model with the official SAS LASSO
# ------------------------------------------------------------

sas_python_check = (
    lasso_scored_2024[
        [
            "product_category_code",
            "P_next_year_log_export_value"
        ]
    ]
    .merge(
        lasso_h1_score[
            [
                "product_category_code",
                "predicted_log_h1"
            ]
        ],
        on="product_category_code",
        validate="one_to_one"
    )
)

assert len(sas_python_check) == 98

pearson_correlation = (
    sas_python_check[
        "P_next_year_log_export_value"
    ]
    .corr(
        sas_python_check["predicted_log_h1"]
    )
)

spearman_correlation = (
    sas_python_check[
        "P_next_year_log_export_value"
    ]
    .corr(
        sas_python_check["predicted_log_h1"],
        method="spearman"
    )
)

# ------------------------------------------------------------
# Aggregate the direct two-year predictions for 2025
# ------------------------------------------------------------

predicted_2025_total = (
    lasso_h2_score["predicted_usd_h2"].sum()
    / 1_000_000
)

predicted_2025_diamonds = (
    lasso_h2_score.loc[
        lasso_h2_score["is_diamond_category"].eq(1),
        "predicted_usd_h2"
    ].sum()
    / 1_000_000
)

predicted_2025_non_diamonds = (
    predicted_2025_total
    - predicted_2025_diamonds
)

# Retrieve the official Statistics Botswana 2025 actuals.
actual_2025_rows = statsbots_annual_actuals_master.loc[
    statsbots_annual_actuals_master[
        "reporting_year"
    ].eq(2025)
]

assert len(actual_2025_rows) == 1, (
    "Exactly one 2025 actual-results row was expected."
)

actual_2025 = actual_2025_rows.squeeze()

# ------------------------------------------------------------
# Construct the 2025 external-validation table
# ------------------------------------------------------------

lasso_2025_external_validation = pd.DataFrame({
    "export_component": [
        "Total exports",
        "Diamond exports (HS 7102)",
        "Non-diamond exports"
    ],

    "baseline_2023_usd_million": [
        baseline_2023_total_usd_million,
        baseline_2023_diamond_usd_million,
        baseline_2023_non_diamond_usd_million
    ],

    "predicted_2025_usd_million": [
        predicted_2025_total,
        predicted_2025_diamonds,
        predicted_2025_non_diamonds
    ],

    "actual_2025_usd_million": [
        actual_2025["total_exports_usd_million"],
        actual_2025["diamond_exports_usd_million"],
        actual_2025["non_diamond_exports_usd_million"]
    ]
})

result = lasso_2025_external_validation

result["forecast_error_usd_million"] = (
    result["predicted_2025_usd_million"]
    - result["actual_2025_usd_million"]
)

result["absolute_error_usd_million"] = (
    result["forecast_error_usd_million"].abs()
)

result["percentage_error_pct"] = (
    100
    * result["forecast_error_usd_million"]
    / result["actual_2025_usd_million"]
)

result["absolute_percentage_error_pct"] = (
    result["percentage_error_pct"].abs()
)

result["predicted_change_from_2023_pct"] = (
    100
    * (
        result["predicted_2025_usd_million"]
        / result["baseline_2023_usd_million"]
        - 1
    )
)

result["actual_change_from_2023_pct"] = (
    100
    * (
        result["actual_2025_usd_million"]
        / result["baseline_2023_usd_million"]
        - 1
    )
)

result["direction_correct"] = (
    np.sign(
        result["predicted_change_from_2023_pct"]
    )
    ==
    np.sign(
        result["actual_change_from_2023_pct"]
    )
)

result.insert(
    0,
    "model",
    "Python direct-horizon LASSO extension"
)

result.insert(
    1,
    "forecast_year",
    2025
)

result["actual_source"] = "Statistics Botswana"

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("Direct-horizon LASSO extension completed.")

print(
    f"SAS/Python h=1 correlation: "
    f"Pearson={pearson_correlation:.4f}; "
    f"Spearman={spearman_correlation:.4f}"
)

print("\nTime-based validation and refit summary:")
display(lasso_horizon_validation)

print("\n2025 external-validation results:")
display(lasso_2025_external_validation)

Direct-horizon LASSO extension completed.
SAS/Python h=1 correlation: Pearson=0.9996; Spearman=0.9993

Time-based validation and refit summary:


,horizon_years,forecast_year,selected_alpha,validation_ase,training_origin_years,validation_origin_years,refit_origin_years,refit_rows
0,1,2024,0.1124,2.7908,2005-2018,2019-2022,2005-2022,1764
1,2,2025,0.0131,3.8195,2005-2017,2018-2021,2005-2021,1666



2025 external-validation results:


,model,forecast_year,export_component,baseline_2023_usd_million,predicted_2025_usd_million,actual_2025_usd_million,forecast_error_usd_million,absolute_error_usd_million,percentage_error_pct,absolute_percentage_error_pct,predicted_change_from_2023_pct,actual_change_from_2023_pct,direction_correct,actual_source
0,Python direct-horizon LASSO extension,2025,Total exports,"5,502.7165","6,109.2104","5,503.5160",605.6944,605.6944,11.0056,11.0056,11.0217,0.0145,True,Statistics Botswana
1,Python direct-horizon LASSO extension,2025,Diamond exports (HS 7102),"4,333.5664","5,390.4891","3,996.2880","1,394.2011","1,394.2011",34.8874,34.8874,24.3892,-7.7829,False,Statistics Botswana
2,Python direct-horizon LASSO extension,2025,Non-diamond exports,"1,169.1501",718.7213,"1,507.2280",-788.5067,788.5067,-52.3150,52.3150,-38.5262,28.9165,False,Statistics Botswana


## Step 12 — Compare the SVM and LASSO models against 2024 actuals
Compares the SAS champion (SVM) and LASSO models on the same three export components. On this real-world test, LASSO was more accurate: its average error was 26.4% versus 45.3% for SVM, and it predicted the direction of change correctly for 2 of 3 components versus 0 of 3 for SVM.

In [17]:
# ============================================================
# STEP 12: Compare SAS SVM and SAS LASSO against 2024 actuals
# Run after Step 10. This creates no new files.
# ============================================================

from pathlib import Path

svm_score_path = Path("table_FINALPROJECTSCOREDATA.csv")

assert svm_score_path.exists(), (
    "Place table_FINALPROJECTSCOREDATA.csv "
    "in the same folder as the notebook."
)

assert "lasso_2024_external_validation" in globals(), (
    "Run Step 10 before running this cell."
)

# ------------------------------------------------------------
# Import and validate the SAS SVM score data
# ------------------------------------------------------------

svm_scored_2024 = pd.read_csv(
    svm_score_path,
    dtype={
        "product_category_code": "string",
        "product_year_id": "string"
    }
)

svm_scored_2024["product_category_code"] = (
    svm_scored_2024["product_category_code"]
    .str.strip()
    .str.zfill(2)
)

required_svm_columns = [
    "product_category_code",
    "reporting_year",
    "is_diamond_category",
    "P_next_year_log_export_value",
    "EM_PREDICTION"
]

missing_svm_columns = [
    column
    for column in required_svm_columns
    if column not in svm_scored_2024.columns
]

assert not missing_svm_columns, (
    f"Required SVM columns are missing: "
    f"{missing_svm_columns}"
)

assert len(svm_scored_2024) == 98, (
    f"Expected 98 SVM score rows; "
    f"found {len(svm_scored_2024)}."
)

assert (
    svm_scored_2024["product_category_code"].nunique()
    == 98
), "Expected 98 unique product categories."

assert svm_scored_2024["reporting_year"].eq(2023).all(), (
    "The SVM score rows are not all based on 2023."
)

assert (
    svm_scored_2024[required_svm_columns]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing values were found in required SVM fields."

# Confirm the two SAS prediction fields agree.
maximum_prediction_difference = (
    svm_scored_2024[
        "P_next_year_log_export_value"
    ]
    - svm_scored_2024["EM_PREDICTION"]
).abs().max()

assert maximum_prediction_difference < 1e-10, (
    "The two SVM prediction columns do not agree."
)

# ------------------------------------------------------------
# Convert the log predictions back to export values
# ------------------------------------------------------------

svm_scored_2024[
    "predicted_2024_export_value_usd"
] = (
    np.expm1(
        svm_scored_2024[
            "P_next_year_log_export_value"
        ]
    )
    .clip(lower=0)
)

# Aggregate the product-category predictions.
svm_total_usd_million = (
    svm_scored_2024[
        "predicted_2024_export_value_usd"
    ].sum()
    / 1_000_000
)

svm_diamond_usd_million = (
    svm_scored_2024.loc[
        svm_scored_2024[
            "is_diamond_category"
        ].eq(1),
        "predicted_2024_export_value_usd"
    ].sum()
    / 1_000_000
)

svm_non_diamond_usd_million = (
    svm_total_usd_million
    - svm_diamond_usd_million
)

# ------------------------------------------------------------
# Use exactly the same baseline and official actual values
# used in the LASSO external-validation table
# ------------------------------------------------------------

reference = (
    lasso_2024_external_validation
    .set_index("export_component")
)

components = [
    "Total exports",
    "Diamond exports (HS 7102)",
    "Non-diamond exports"
]

svm_2024_external_validation = pd.DataFrame({
    "model": "SAS Support Vector Machine",

    "forecast_year": 2024,

    "export_component": components,

    "baseline_2023_usd_million": (
        reference.loc[
            components,
            "baseline_2023_usd_million"
        ].to_numpy()
    ),

    "predicted_2024_usd_million": [
        svm_total_usd_million,
        svm_diamond_usd_million,
        svm_non_diamond_usd_million
    ],

    "actual_2024_usd_million": (
        reference.loc[
            components,
            "actual_2024_usd_million"
        ].to_numpy()
    )
})

# ------------------------------------------------------------
# Calculate identical evaluation measures
# ------------------------------------------------------------

svm_result = svm_2024_external_validation

svm_result["forecast_error_usd_million"] = (
    svm_result["predicted_2024_usd_million"]
    - svm_result["actual_2024_usd_million"]
)

svm_result["absolute_error_usd_million"] = (
    svm_result["forecast_error_usd_million"].abs()
)

svm_result["percentage_error_pct"] = (
    100
    * svm_result["forecast_error_usd_million"]
    / svm_result["actual_2024_usd_million"]
)

svm_result["absolute_percentage_error_pct"] = (
    svm_result["percentage_error_pct"].abs()
)

svm_result["predicted_change_from_2023_pct"] = (
    100
    * (
        svm_result["predicted_2024_usd_million"]
        / svm_result["baseline_2023_usd_million"]
        - 1
    )
)

svm_result["actual_change_from_2023_pct"] = (
    100
    * (
        svm_result["actual_2024_usd_million"]
        / svm_result["baseline_2023_usd_million"]
        - 1
    )
)

svm_result["direction_correct"] = (
    np.sign(
        svm_result[
            "predicted_change_from_2023_pct"
        ]
    )
    ==
    np.sign(
        svm_result[
            "actual_change_from_2023_pct"
        ]
    )
)

svm_result["actual_source"] = (
    "Statistics Botswana"
)

# ------------------------------------------------------------
# Combine SVM and LASSO results
# ------------------------------------------------------------

comparison_columns = [
    "model",
    "forecast_year",
    "export_component",
    "baseline_2023_usd_million",
    "predicted_2024_usd_million",
    "actual_2024_usd_million",
    "forecast_error_usd_million",
    "absolute_error_usd_million",
    "percentage_error_pct",
    "absolute_percentage_error_pct",
    "predicted_change_from_2023_pct",
    "actual_change_from_2023_pct",
    "direction_correct",
    "actual_source"
]

lasso_svm_2024_external_comparison = (
    pd.concat(
        [
            lasso_2024_external_validation[
                comparison_columns
            ],
            svm_2024_external_validation[
                comparison_columns
            ]
        ],
        ignore_index=True
    )
    .sort_values(
        ["export_component", "model"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Create compact comparison summaries
# ------------------------------------------------------------

total_export_summary = (
    lasso_svm_2024_external_comparison.loc[
        lasso_svm_2024_external_comparison[
            "export_component"
        ].eq("Total exports"),
        [
            "model",
            "predicted_2024_usd_million",
            "actual_2024_usd_million",
            "forecast_error_usd_million",
            "absolute_percentage_error_pct",
            "direction_correct"
        ]
    ]
    .sort_values(
        "absolute_percentage_error_pct"
    )
    .reset_index(drop=True)
)

model_summary = (
    lasso_svm_2024_external_comparison
    .groupby("model", as_index=False)
    .agg(
        mean_absolute_percentage_error_across_3_components=(
            "absolute_percentage_error_pct",
            "mean"
        ),
        directions_correct_out_of_3=(
            "direction_correct",
            "sum"
        )
    )
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("SAS SVM score file validated.")

print(
    f"Scored product categories: "
    f"{len(svm_scored_2024)}"
)

print(
    "Maximum difference between SVM prediction columns: "
    f"{maximum_prediction_difference:.12f}"
)

print("\nTotal-export comparison:")
display(total_export_summary)

print("\nFull external-validation comparison:")
display(lasso_svm_2024_external_comparison)

print("\nThree-component descriptive summary:")
display(model_summary)

SAS SVM score file validated.
Scored product categories: 98
Maximum difference between SVM prediction columns: 0.000000000000

Total-export comparison:


,model,predicted_2024_usd_million,actual_2024_usd_million,forecast_error_usd_million,absolute_percentage_error_pct,direction_correct
0,SAS LASSO Regression,"4,803.3966","4,664.9797",138.4169,2.9671,True
1,SAS Support Vector Machine,"6,423.3043","4,664.9797","1,758.3246",37.6920,False



Full external-validation comparison:


,model,forecast_year,export_component,baseline_2023_usd_million,predicted_2024_usd_million,actual_2024_usd_million,forecast_error_usd_million,absolute_error_usd_million,percentage_error_pct,absolute_percentage_error_pct,predicted_change_from_2023_pct,actual_change_from_2023_pct,direction_correct,actual_source
0,SAS LASSO Regression,2024,Diamond exports (HS 7102),"4,333.5664","4,086.8527","3,234.8918",851.9610,851.9610,26.3366,26.3366,-5.6931,-25.3527,True,Statistics Botswana
1,SAS Support Vector Machine,2024,Diamond exports (HS 7102),"4,333.5664","5,428.2993","3,234.8918","2,193.4075","2,193.4075",67.8047,67.8047,25.2617,-25.3527,False,Statistics Botswana
2,SAS LASSO Regression,2024,Non-diamond exports,"1,169.1501",716.5439,"1,430.0879",-713.5440,713.5440,-49.8951,49.8951,-38.7124,22.3186,False,Statistics Botswana
3,SAS Support Vector Machine,2024,Non-diamond exports,"1,169.1501",995.0050,"1,430.0879",-435.0829,435.0829,-30.4235,30.4235,-14.8950,22.3186,False,Statistics Botswana
4,SAS LASSO Regression,2024,Total exports,"5,502.7165","4,803.3966","4,664.9797",138.4169,138.4169,2.9671,2.9671,-12.7086,-15.2241,True,Statistics Botswana
5,SAS Support Vector Machine,2024,Total exports,"5,502.7165","6,423.3043","4,664.9797","1,758.3246","1,758.3246",37.6920,37.6920,16.7297,-15.2241,False,Statistics Botswana



Three-component descriptive summary:


,model,mean_absolute_percentage_error_across_3_components,directions_correct_out_of_3
0,SAS LASSO Regression,26.3996,2
1,SAS Support Vector Machine,45.3067,0
